# TBP — Primera prueba con series reales de Argentina

Este notebook calcula el TBP por cohorte (1950–2035) con las series de `datos/procesados/` (ver `datos/README.md`). Es una **prueba exploratoria**: sirve para ver si los datos se comportan, no para concluir.

**Qué es dato y qué es supuesto** (todo queda a la vista):

| Variable | Valor | Origen | Tipo |
|---|---|---|---|
| $N_t$ | nacimientos | DEIS (≤2024); ONU proyección (2025+) | observado / proyectado |
| $1-\mu$ | supervivencia de la **cohorte** hasta 65 | ONU WPP 2024, tabla de mortalidad (producto de $p_x$ por edad y año calendario) | estimado |
| $J_{t+65}$ | población 65+ en $t+65$ | ONU WPP 2024 | proyectado |
| $E_r$ | esperanza de vida a los 65 en $t+65$ | ONU WPP 2024 (de período) | proyectado |
| $\tau$ | 11 % personal + 10,77 % patronal = **21,77 %** | Anuario ANSES 2008–2023, p. 39 | normativa (sin aportes a PAMI/obra social) |
| $\rho$ | **40,3 %** (haber medio SIPA / salario imponible medio, dic-2023) | Anuario ANSES, p. 91 | observado, **constante** |
| $\bar A$ | **14,2** (obs.) o **30** (legal) años | ANSES p. 46 (varones 64 años: 15,8; mujeres 59 años: 12,6; promedio simple) / requisito legal | **supuesto — escenario** |

> ⚠️ $\bar A$ observado está **sesgado a la baja**: ANSES solo cuenta aportes desde jul-1994 y a personas con ≥12 meses de aporte. El 30 es el requisito legal (techo). La verdad está entre ambos. **Por eso el nivel del TBP no es interpretable todavía; la forma de la curva y su tendencia sí.**

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
try: display
except NameError: display = print

# --- Carga de datos -------------------------------------------------------
# Orden de busqueda: 1) carpeta del repo (../datos/procesados), 2) Drive (opcional), 3) descarga desde GitHub.
# En Colab con Drive: descomentar las dos lineas siguientes para montarlo.
# from google.colab import drive; drive.mount('/content/drive')
ARCHIVOS = ["series_tbp_extraidas.csv", "un_wpp2024_argentina_tabla_mortalidad_abreviada.csv"]
# Repositorio del proyecto
RAW_BASE = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDIDATOS = ["../datos/procesados", "datos/procesados",
              "/content/drive/MyDrive/JUANDE/CURSAR/06. Taller de indicadores/datos_tbp",
              "/content/drive/MyDrive/datos_tbp", "/content/datos_tbp", "."]
DATA = next((p for p in CANDIDATOS if os.path.exists(os.path.join(p, "series_tbp_extraidas.csv"))), None)
if DATA is None:   # no hay copia local: descargar desde GitHub (raw)
    DATA = "datos_tbp_descargados"; os.makedirs(DATA, exist_ok=True)
    for f in ARCHIVOS:
        urllib.request.urlretrieve(f"{RAW_BASE}/{f}", os.path.join(DATA, f))
print("Datos en:", os.path.abspath(DATA))

# Las salidas (CSV y PNG) se escriben en ./salidas_tbp para no pisar los datos versionados.
OUT = "salidas_tbp"; os.makedirs(OUT, exist_ok=True)

s = pd.read_csv(f"{DATA}/series_tbp_extraidas.csv").set_index("year")
mort = pd.read_csv(f"{DATA}/un_wpp2024_argentina_tabla_mortalidad_abreviada.csv", low_memory=False)
mort = mort[mort.Sex == "Total"]
display(s.loc[[1950, 1990, 2010, 2024, 2040, 2091], ["deis_nacimientos", "un_births_miles", "un_e65", "un_pob_65mas_miles"]])

## 1. Supervivencia de la cohorte (1−μ)

Para la cohorte nacida en $t$ se multiplica la probabilidad de sobrevivir cada grupo de edad ($0, 1\text{–}4, 5\text{–}9,\dots,60\text{–}64$) usando la tabla de mortalidad del **año calendario en que ese grupo vive**. Es más correcto que usar la tabla del año de nacimiento ("de período"), que ignora que la mortalidad va bajando.

In [ ]:
px = mort.pivot_table(index="Time", columns="AgeGrpStart", values="px")
grp = mort[mort.Time == 2000].sort_values("AgeGrpStart")[["AgeGrpStart", "AgeGrpSpan"]]
grp = grp[grp.AgeGrpStart < 65]

def surv65_cohorte(t):
    S = 1.0
    for a, sp in grp.itertuples(index=False):
        y = int(np.floor(t + a + sp/2))
        if y > px.index.max() or y < px.index.min():
            return np.nan
        S *= px.loc[y, a]
    return S

print("Ejemplo cohorte 2000: 1-mu =", round(surv65_cohorte(2000), 4), "| tabla de período del año 2000 (ignora mejoras futuras):", round(s.un_S65_periodo[2000], 4))

## 2. Cálculo por cohorte

$\tau$ y $\rho$ son constantes (los datos de ANSES no dan serie anual utilizable, ver `anses_valores_extraidos.csv`). **Todo el movimiento de la curva viene de la demografía** ($N_t$, $1-\mu$, $J$, $E_r$).

In [ ]:
TAU = 0.11 + 0.1077     # Anuario ANSES p.39 (aporte personal 11% + contribución patronal SIPA 10,77% actividades inc. a)
RHO = 0.403             # Anuario ANSES p.91: haber medio SIPA / salario imponible medio, dic-2023
A_OBS, A_LEGAL = 14.2, 30.0

rows = []
for t in range(1950, 2036):
    y = t + 65
    N_deis = s.deis_nacimientos.get(t, np.nan)
    N_un = s.un_births_miles[t] * 1000
    rows.append(dict(cohorte=t, anio_65=y,
        N_t=N_deis if not np.isnan(N_deis) else N_un,
        fuente_N="DEIS" if not np.isnan(N_deis) else "ONU proyectado",
        N_un=N_un, S65=surv65_cohorte(t),
        J=s.un_pob_65mas_miles[y]*1000, E65=s.un_e65[y], pob_edad65=s.un_pob_edad65_miles[y]*1000))
d = pd.DataFrame(rows)

for nom, A in [("obs", A_OBS), ("legal", A_LEGAL)]:
    d[f"TBP_A_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*d.E65*RHO)     # fórmula original del grupo
    d[f"TBP_C_{nom}"] = d.N_t*d.S65*A*TAU/(d.J*RHO)           # en años (sin E_r)
    d[f"TBP_B_{nom}"] = A*TAU/(d.E65*RHO)                     # cohorte pura, sin N
d["D"] = d.N_t*d.S65/d.J                                       # factor demográfico

d.to_csv(f"{OUT}/tbp_argentina_cohortes_1950_2035.csv", index=False)
display(d[d.cohorte.isin([1950,1975,1990,2000,2010,2015,2020,2024,2030,2035])][
    ["cohorte","N_t","fuente_N","S65","J","E65","D","TBP_A_obs","TBP_A_legal","TBP_C_legal","TBP_B_legal"]])

## 3. Grupo de control con datos reales

Tres chequeos que **no dependen** de los supuestos de $\bar A$, $\tau$ ni $\rho$:

1. **Identidades algebraicas:** $TBP_A = D\cdot B$ y $TBP_A = TBP_C/E_r$.
2. **Consistencia demográfica:** la población de exactamente 65 años en $t+65$ (ONU) debería ser ≈ $N_t \cdot (1-\mu)$ (nacidos que sobreviven; la diferencia es migración y error de las fuentes).
3. **Consistencia entre fuentes:** nacimientos DEIS vs ONU.

In [ ]:
# 1) identidades
B = d.TBP_B_legal
assert np.allclose(d.TBP_A_legal, d.D * B)
assert np.allclose(d.TBP_A_legal, d.TBP_C_legal / d.E65)
print("✅ Identidades TBP_A = D×B y TBP_A = TBP_C/E_r: OK en las 86 cohortes")

# 2) cohorte reconstruida vs población observada/proyectada a los 65
d["ratio_pob65_vs_NS"] = d.pob_edad65 / (d.N_t * d.S65)
r = d.ratio_pob65_vs_NS
print(f"\n2) población(65 años en t+65) / (N_t × S65): media {r.mean():.3f} | min {r.min():.3f} | max {r.max():.3f}")
fuera = d[(r < 0.85) | (r > 1.20)][["cohorte", "fuente_N", "ratio_pob65_vs_NS"]]
print("   Cohortes fuera de [0.85, 1.20]:", "ninguna" if fuera.empty else "")
if not fuera.empty: display(fuera)

# 3) DEIS vs ONU
c = d[d.fuente_N == "DEIS"].copy()
c["ONU/DEIS"] = c.N_un / c.N_t
print(f"\n3) Nacimientos ONU/DEIS ({c.cohorte.min()}–{c.cohorte.max()}): media {c['ONU/DEIS'].mean():.3f} | min {c['ONU/DEIS'].min():.3f} | max {c['ONU/DEIS'].max():.3f}")
display(c.loc[c.cohorte.isin([1950,1980,2000,2015,2020,2022,2023,2024]), ["cohorte", "N_t", "N_un", "ONU/DEIS"]])

## 4. Gráficos

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.6))
ax[0].plot(d.cohorte, d.N_t/1e3, label="Usado (DEIS ≤2024, ONU después)")
ax[0].plot(d.cohorte, d.N_un/1e3, "--", label="ONU WPP 2024")
ax[0].axvline(2024.5, color="gray", ls=":"); ax[0].set_title("Nacimientos (miles)"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)

ax[1].plot(d.cohorte, d.D, color="tab:green")
ax[1].axvline(2024.5, color="gray", ls=":"); ax[1].set_title("D = N·(1−μ) / J₆₅₊ (factor demográfico)"); ax[1].grid(alpha=.3)

ax[2].plot(d.cohorte, d.TBP_A_legal, label="TBP_A  (Ā = 30, techo)")
ax[2].plot(d.cohorte, d.TBP_A_obs, label="TBP_A  (Ā = 14,2, piso sesgado)")
ax[2].plot(d.cohorte, d.TBP_B_legal, "--", label="TBP_B (cohorte pura, Ā = 30)")
ax[2].axhline(1, color="gray", ls=":"); ax[2].axvline(2024.5, color="gray", ls=":")
ax[2].set_title("TBP por cohorte"); ax[2].legend(fontsize=8); ax[2].grid(alpha=.3)
for a in ax: a.set_xlabel("año de nacimiento de la cohorte")
plt.tight_layout(); plt.savefig(f"{OUT}/tbp_argentina_graficos.png", dpi=130); plt.show()

## 5. Qué muestran los datos (lectura preliminar)

* **Caída de la natalidad real:** de ~756 mil (2010) a 413 mil (2024). La cohorte 2024 tiene el factor demográfico $D$ más bajo de toda la serie.
* **Rebote artificial después de 2024:** la ONU proyecta ~507 mil nacimientos en 2024 y ~520 mil hacia 2030. Eso **no refleja** la caída registrada por DEIS. El repunte de las cohortes 2025–2035 es un efecto del supuesto ONU, **no un dato**: hay que reemplazarlo con un escenario propio de natalidad.
* **Longevidad:** $E_{65}$ pasa de 17,9 (cohorte 1950) a ~24,6 (cohorte 2035). Eso empuja el TBP hacia abajo por dos vías (en $J$ y en $E_r$), el problema de doble conteo detectado en la auditoría.
* **La curva de $TBP_B$** (sin natalidad) baja suavemente por longevidad; la de $TBP_A$ baja mucho más y tiene la caída de natalidad encima. La diferencia entre ambas es el aporte de la natalidad.
* **Todavía no se puede afirmar:** niveles absolutos ($\bar A$ incierto), umbral "TBP=1", ni proyecciones más allá de 2024 basadas en natalidad ONU.

## 6. Fe de erratas (02-10-2026)

Las secciones 1 a 5 de arriba se dejan **tal como se presentaron originalmente**. Una revisión posterior detectó los puntos siguientes; se corrigen acá, al final, para que quede registro de la evolución del trabajo.

| # | Dónde | Decía | Corrección | Por qué |
|---|---|---|---|---|
| 1 | Sección 5, *Longevidad* | $E_{65}$ pasa de 17,9 a ~24,6 | $E_{65}$ va de **16,1** (cohorte 1956) a 24,6 (cohorte 2035); 17,9 corresponde a la cohorte 1950 | La cohorte 1956 cumple 65 en 2021 y $E_r$ sale de la tabla de *período* de ese año (COVID): pico atípico, no un error del cálculo |
| 2 | Sección 5, *curva de $TBP_B$* | "baja suavemente"; (en el resumen se citaron rangos 0,91→0,66 y 0,02–0,06) | $TBP_B$ va de **0,66 a 1,01**; $TBP_A$ de **0,018 a 0,066**; $TBP_C$ de 0,43 a 1,13 años. El máximo de los tres está en la cohorte 1956 | El resumen miró solo los extremos y omitió ese pico |
| 3 | Introducción y gráfico (*"Ā = 30, techo"*) | 30 años = techo legal | 30 años es el **requisito mínimo** para jubilarse, **no un máximo**: un promedio con ceros puede quedar muy por debajo | Error de concepto |
| 4 | Sección 5, natalidad | (no se mencionaba) | $N_t$ salta de 413.135 (DEIS 2024) a 508.067 (ONU 2025) | Es el cambio de fuente, no un hecho demográfico: el rebote posterior es un artefacto del supuesto ONU |

La celda siguiente **recalcula** esas cifras desde los datos, para que cualquiera pueda verificarlas.

In [ ]:
c = d.set_index("cohorte")
print(f"E_65: mín = {d.E65.min():.2f} (cohorte {int(d.loc[d.E65.idxmin(), 'cohorte'])}) | máx = {d.E65.max():.2f} (cohorte {int(d.loc[d.E65.idxmax(), 'cohorte'])}) | cohorte 1950 = {c.E65[1950]:.2f}")
print(f"TBP_B (Ā=30): {d.TBP_B_legal.min():.3f} a {d.TBP_B_legal.max():.3f} (máx. en la cohorte {int(d.loc[d.TBP_B_legal.idxmax(), 'cohorte'])})")
print(f"TBP_A (Ā=30): {d.TBP_A_legal.min():.4f} a {d.TBP_A_legal.max():.4f}")
print(f"TBP_C (Ā=30, años): {d.TBP_C_legal.min():.3f} a {d.TBP_C_legal.max():.3f}")
print(f"N_t usado: cohorte 2024 = {c.N_t[2024]:,.0f} ({c.fuente_N[2024]}) -> cohorte 2025 = {c.N_t[2025]:,.0f} ({c.fuente_N[2025]})")

## 7. Solución 1 — Sensibilidad por bandas del parámetro $\bar A$

**Opción 1 de la propuesta del grupo.** En vez de fijar $\bar A$ en un único valor, corremos el modelo para $\bar A \in \{15, 20, 25, 30\}$ años y graficamos una **banda de incertidumbre**.

Contexto: hoy $\bar A$ tiene dos escenarios: 14,2 (observado ANSES, **sesgado a la baja**: ANSES cuenta aportes solo desde jul-1994 y a personas con ≥12 meses de aporte) y 30 (requisito legal de referencia: es el mínimo para jubilarse, no un máximo). La verdad está entre ambos, pero no sabemos dónde.

$$TBP_A = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,E_r\,\rho},\qquad TBP_C = \frac{N_t(1-\mu)\,\bar A\,\tau}{J_{t+65}\,\rho}\ \text{(años)},\qquad TBP_B=\frac{\bar A\,\tau}{E_r\,\rho}\ \text{(cohorte pura)}$$

Constantes: $\tau = 0{,}2177$ (11 % personal + 10,77 % patronal; Anuario ANSES p. 39) y $\rho = 0{,}403$ (tasa de sustitución ANSES, dic-2023, p. 91). Todo lo demás sale del CSV `tbp_argentina_cohortes_1950_2035.csv`; no se inventa ningún dato.

> **Qué NO hace este notebook:** no corrige el sesgo de $\bar A$ ni lo estima. Es una forma honesta de *mostrar* la incertidumbre de nivel. La estimación va por la Opción 2 (calibración con EPH) y la Opción 3 ($\bar A_t$ dinámico).

### 7.2 Parámetros y cálculo para cada escenario de $\bar A$

In [ ]:
OUT = globals().get("OUT", DATA)   # carpeta de salidas del notebook
TAU = 0.11 + 0.1077     # Anuario ANSES p.39
RHO = 0.403             # Anuario ANSES p.91 (dic-2023), constante
A_ESCENARIOS = [15, 20, 25, 30]     # parametrizable
CORTE = 2024.5          # DEIS observado (<=2024) vs ONU proyectado (>=2025)

# N_t * S65 / J = D (factor demografico, ya en el CSV); lo reconstruimos desde las columnas base
d["D_calc"] = d.N_t * d.S65 / d.J
assert np.allclose(d.D_calc, d.D)

filas = []
for A in A_ESCENARIOS:
    t = pd.DataFrame({"cohorte": d.cohorte, "anio_65": d.anio_65, "fuente_N": d.fuente_N, "A": A})
    t["TBP_A"] = d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    t["TBP_C"] = d.N_t * d.S65 * A * TAU / (d.J * RHO)
    t["TBP_B"] = A * TAU / (d.E65 * RHO)
    for v in ["TBP_A", "TBP_B", "TBP_C"]:
        base = t.loc[t.cohorte == 1950, v].iloc[0]
        t[v + "_idx"] = 100 * t[v] / base          # indice base cohorte 1950 = 100
    filas.append(t)
sens = pd.concat(filas, ignore_index=True)
sens = sens.rename(columns={"A": "A_barra"})
sens.head()

### 7.3 Test de control (asserts)

1. **Consistencia con el CSV existente:** nuestro cálculo debe reproducir `TBP_*_legal` (Ā = 30) y `TBP_*_obs` (Ā = 14,2) del CSV.
2. **Linealidad en $\bar A$:** $TBP(2\bar A) = 2\cdot TBP(\bar A)$.
3. **Invarianza del índice normalizado:** el índice base 1950 = 100 es idéntico en todos los escenarios.

In [ ]:
def tbp(A, tipo):
    if tipo == "A": return d.N_t * d.S65 * A * TAU / (d.J * d.E65 * RHO)
    if tipo == "C": return d.N_t * d.S65 * A * TAU / (d.J * RHO)
    if tipo == "B": return A * TAU / (d.E65 * RHO)

# 1) reproduce el CSV existente
for tipo in "ABC":
    assert np.allclose(tbp(30.0, tipo), d[f"TBP_{tipo}_legal"])
    assert np.allclose(tbp(14.2, tipo), d[f"TBP_{tipo}_obs"])
print("OK 1) reproduce TBP_*_legal (A=30) y TBP_*_obs (A=14,2) del CSV")

# 2) linealidad
for tipo in "ABC":
    for A in [7.5, 10, 12.5, 15]:
        assert np.allclose(tbp(2 * A, tipo), 2 * tbp(A, tipo))
print("OK 2) linealidad: TBP(2A) = 2*TBP(A) para A, B y C")

# 3) indice normalizado identico entre escenarios
for v in ["TBP_A_idx", "TBP_B_idx", "TBP_C_idx"]:
    piv = sens.pivot(index="cohorte", columns="A_barra", values=v)
    ref = piv.iloc[:, 0]
    for c in piv.columns:
        assert np.allclose(piv[c], ref)
print("OK 3) el indice normalizado (base 1950=100) es identico entre escenarios")

### 7.4 Hallazgo clave: $\bar A$ cambia el **nivel**, no la **historia**

Como $\bar A$ entra en el numerador como factor multiplicativo y no depende de la cohorte $t$ (en este modelo es una constante), el TBP es **lineal homogéneo** en $\bar A$:

$$TBP_C(t;\bar A) = \bar A\cdot\underbrace{\frac{N_t(1-\mu_t)\,\tau}{J_{t+65}\,\rho}}_{k_C(t)\ \text{(no depende de }\bar A)}$$

Entonces, para dos escenarios cualesquiera $\bar A_1,\bar A_2$ y dos cohortes $t,s$:

$$\frac{TBP_C(t;\bar A_1)}{TBP_C(s;\bar A_1)}=\frac{\bar A_1\,k_C(t)}{\bar A_1\,k_C(s)}=\frac{k_C(t)}{k_C(s)}=\frac{\bar A_2\,k_C(t)}{\bar A_2\,k_C(s)}=\frac{TBP_C(t;\bar A_2)}{TBP_C(s;\bar A_2)}$$

$\bar A$ se cancela. Lo mismo vale para $TBP_A$ ($k_A=k_C/E_r$) y $TBP_B$. Por lo tanto:

* El **índice normalizado** (cohorte 1950 = 100) es **idéntico** para cualquier $\bar A$: $\text{Idx}(t)=100\,k(t)/k(1950)$.
* Las **variaciones porcentuales**, los **cocientes entre cohortes**, los **puntos de inflexión** y las **tendencias relativas** no dependen de $\bar A$.
* Lo que sí cambia con $\bar A$ es el **nivel**: la curva se estira verticalmente en un factor $\bar A_2/\bar A_1$ (entre $\bar A=15$ y $\bar A=30$, un factor 2).

**Consecuencia para la presentación:** la incertidumbre sobre $\bar A$ es incertidumbre de *nivel*, no de *comportamiento*. La historia (caída por natalidad y longevidad, ver abajo) es robusta; el valor absoluto, no.

> **Advertencia sobre la parte proyectada:** a partir de la cohorte 2025 los nacimientos $N_t$ son proyección ONU, no dato DEIS. La ONU proyecta ~507 mil nacimientos en 2024 vs ~413 mil observados por DEIS; el **rebote posterior a 2024 es un artefacto del supuesto ONU**, no un dato. No debe leerse como recuperación.
>
> **Sobre el nivel:** no interpretamos "TBP = 1" como umbral fiscal; el nivel depende de $\bar A$, de $\tau$ y de $\rho$ constantes, y no de un análisis de sostenibilidad.

### 7.5 Gráficos

In [ ]:
COL = plt.cm.viridis(np.linspace(0.1, 0.85, len(A_ESCENARIOS)))

def banda(ax, var, titulo, ylabel):
    piv = sens.pivot(index="cohorte", columns="A_barra", values=var)
    ax.fill_between(piv.index, piv.min(axis=1), piv.max(axis=1), color="tab:blue", alpha=0.15,
                    label=f"banda min-max (A = {min(A_ESCENARIOS)} a {max(A_ESCENARIOS)})")
    for c, A in zip(COL, A_ESCENARIOS):
        ax.plot(piv.index, piv[A], color=c, lw=1.6, label=f"A = {A}")
    ax.axvline(CORTE, color="gray", ls="--", lw=1)
    ax.text(CORTE + 0.5, ax.get_ylim()[1] * 0.97, "2024/2025: DEIS | ONU proyectado",
            fontsize=7.5, color="gray", va="top")
    ax.set_title(titulo, fontsize=10); ax.set_ylabel(ylabel); ax.set_xlabel("año de nacimiento de la cohorte")
    ax.grid(alpha=.3); ax.legend(fontsize=7.5, loc="upper right", bbox_to_anchor=(1, 0.88))

fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
banda(ax[0], "TBP_C", "TBP_C (versión recomendada, en años)\nbanda de incertidumbre de NIVEL", "TBP_C (años)")
banda(ax[1], "TBP_A", "TBP_A (fórmula original, con $E_r$)\nbanda de incertidumbre de NIVEL", "TBP_A")

# Panel (b): indice normalizado
for c, A in zip(COL, A_ESCENARIOS):
    ss = sens[sens.A_barra == A]
    ax[2].plot(ss.cohorte, ss.TBP_C_idx, color=c, lw=3.5 - 0.7 * A_ESCENARIOS.index(A), alpha=.8, label=f"A = {A}")
ax[2].axvline(CORTE, color="gray", ls="--", lw=1)
ax[2].axhline(100, color="gray", ls=":", lw=.8)
ax[2].set_title("Índice normalizado de TBP_C (cohorte 1950 = 100)\nlas 4 curvas se superponen: Ā no cambia la forma", fontsize=10)
ax[2].set_xlabel("año de nacimiento de la cohorte"); ax[2].set_ylabel("índice (1950 = 100)")
ax[2].grid(alpha=.3); ax[2].legend(fontsize=7.5)
ax[2].text(CORTE + 0.5, ax[2].get_ylim()[1] * 0.97, "post-2024: supuesto ONU\n(rebote = artefacto)", fontsize=7.5, color="gray", va="top")

fig.suptitle("Sensibilidad al parámetro Ā (años de aporte promedio): cambia el nivel, no la historia", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(f"{OUT}/tbp_sensibilidad_A.png", dpi=130)
plt.show()

### 7.6 Resultados: rango de $TBP_C$ por escenario en cohortes seleccionadas

In [ ]:
sel = [1950, 1975, 1990, 2000, 2010, 2024]
tabla = sens[sens.cohorte.isin(sel)].pivot(index="cohorte", columns="A_barra", values="TBP_C").round(3)
tabla["max/min"] = (tabla[max(A_ESCENARIOS)] / tabla[min(A_ESCENARIOS)]).round(2)
display(tabla)

idx = sens[(sens.A_barra == A_ESCENARIOS[0]) & sens.cohorte.isin(sel)].set_index("cohorte")["TBP_C_idx"].round(1)
print("Índice TBP_C (1950=100), igual en todos los escenarios:")
print(idx.to_string())

### 7.7 Exportar

`tbp_sensibilidad_A.csv`: una fila por cohorte × escenario, con TBP_A, TBP_B, TBP_C en nivel y como índice base 1950 = 100.

In [ ]:
sens.to_csv(f"{OUT}/tbp_sensibilidad_A.csv", index=False)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.csv", sens.shape)
print("Escrito:", f"{OUT}/tbp_sensibilidad_A.png")

### 7.8 Lectura y límites

* **Robusto (no depende de Ā):** forma de la curva, caída relativa entre cohortes, tendencias. Se comunica con el índice normalizado.
* **Incierto (sí depende de Ā):** el nivel. Se comunica con la banda 15–30 (factor 2 entre extremos).
* **No resuelto:** el sesgo de $\bar A$. Las cuatro cifras 15/20/25/30 son una grilla ilustrativa, no una distribución de probabilidad: la banda no es un intervalo de confianza.
* **Post-2024:** el rebote es un artefacto del supuesto ONU de nacimientos.
* **Próximos pasos:** Opción 2 (calibrar $\bar A$ con EPH) para acotar el nivel; Opción 3 ($\bar A_t$ dinámico por cohorte) que, a diferencia de esta, **sí** puede cambiar la forma de la curva.

## 8. Solución 2 — Calibración de Ā con la EPH (método; sin datos reales todavía)

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">
DATOS SINTÉTICOS — no son Argentina<br>(las celdas de prueba usan un dataset inventado; sirven solo para verificar el código)
</h1>

**Objetivo.** Estimar Ā (años de aporte promedio por persona de la cohorte, con ceros incluidos) como

Ā_c = Σ_{a=18}^{64} S_c(a) · d_c(a)

donde S_c(a) = P(sobrevivir hasta la edad a) y d_c(a) = P(aportar a la edad a | vivo), aproximada con la **tasa de empleo registrado** de la EPH tratada como pseudo-panel por año de nacimiento (cohorte = ANO4 − CH06) y edad.

**Estado:** este notebook NO produce ningún valor de Ā para Argentina hasta que se coloquen los microdatos reales de la EPH en `datos/eph/` (ver `datos/eph/README.md` y `docs/calibracion_A_EPH.md`). Sin esos archivos termina limpio con el mensaje "faltan datos".

Estructura: (1) configuración y funciones, (2) prueba con datos sintéticos + asserts, (3) prueba del lector de archivos, (4) corrida real (solo si hay datos).

In [ ]:
# ============ 1. CONFIGURACIÓN Y FUNCIONES ============
import os, re, io, glob, zipfile, tempfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)          # 47 edades simples
RUTAS_CANDIDATAS = ["datos/eph", "../datos/eph", "../../datos/eph"]
RUTAS_S65 = ["datos/procesados/tbp_argentina_cohortes_1950_2035.csv",
             "../datos/procesados/tbp_argentina_cohortes_1950_2035.csv"]
MIN_N_CELDA = 20        # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3  # mínimo de edades observadas para estimar el efecto de una cohorte
GOMPERTZ_B = 0.085      # pendiente de la mortalidad adulta (supuesto; ver docs)
# Regla para cuentapropistas/patrones (la base individual NO trae una variable inequívoca de
# aporte de independientes: a verificar contra el diseño de registro). Opciones:
#   "solo_asalariados": independientes cuentan como NO aportantes (cota inferior)
#   "pp07i":            independientes aportan si PP07I == 1 (uso a verificar; ver docs)
#   "escenario":        independientes aportan con probabilidad IND_SHARE (supuesto explícito)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}

def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))

# ---------- (a) lectura de archivos EPH ----------
def find_eph_files(rutas=RUTAS_CANDIDATAS):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip) en las rutas candidatas."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(r"usu_individual.*\.(txt|csv)$", b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(r"usu_individual.*\.(txt|csv)$", m.lower()):
                            out.append({"path": p, "member": m})
    return out

def _read_csv_flex(handle_or_path):
    df = pd.read_csv(handle_or_path, sep=";", usecols=lambda c: c.strip().upper() in NEEDED,
                     low_memory=False, encoding="latin-1")
    return df

def read_individual(src):
    """Lee UNA base individual trimestral (solo columnas necesarias). Devuelve DataFrame con columnas en MAYÚSCULAS."""
    if src["member"] is None:
        df = _read_csv_flex(src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as f:
            df = _read_csv_flex(io.TextIOWrapper(f, encoding="latin-1"))
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = {"ANO4", "CH06", "ESTADO", "PONDERA"} - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df

def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)

# ---------- (b) pseudo-panel: densidad por edad x cohorte ----------
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Filtra 18-64 con entrevista individual realizada y construye 'reg' (prob. de estar aportando).
    Asalariado (CAT_OCUP==3): reg = (PP07H==1). Independiente (1,2): según ind_rule. Resto: 0."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    reg[asal.values] = (d.loc[asal, "PP07H"] == 1).astype(float).values   # Ns/Nr (9) o 2 => no registrado
    if ind_rule == "pp07i":
        reg[indep.values] = (d.loc[indep, "PP07I"] == 1).astype(float).values
    elif ind_rule == "escenario":
        if ind_share is None or not (0 <= ind_share <= 1):
            raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
        reg[indep.values] = float(ind_share)
    elif ind_rule != "solo_asalariados":
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)     # error de +-1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]

def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    g = g[g["n"] >= min_n].reset_index(drop=True)
    return g

# ---------- modelo logit aditivo cohorte + edad (supuesto: perfil por edad estable entre cohortes) ----------
def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(1 / (1 + np.exp(-eta)), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}

def _sig(x): return 1 / (1 + np.exp(-x))

def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes sin datos suficientes (jóvenes o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)

def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo."""
    f = fit["f"].reindex(EDADES)
    f = f.interpolate(limit_direction="both")           # edades sin ninguna observación: interpola/extiende el perfil
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)

# ---------- (c) supervivencia y Ā ----------
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64, Gompertz con pendiente b calibrado para que S(65)=s65 (por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))

def abar_por_cohorte(dens, s65):
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S

def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """df individual EPH -> tabla por cohorte. Idéntico para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)})
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}

def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera sum S(a) (densidad=1)"
    # monotonía: con la misma densidad, mayor S65 => mayor A_bar
    s_alta = (t["S65"] + 0.02).clip(upper=0.999)
    a2, _ = abar_por_cohorte(dens, s_alta)
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True
print("Funciones cargadas.")

### 8.2 Prueba con DATASET SINTÉTICO

<h1 style="color:#b00020;border:4px solid #b00020;padding:12px;text-align:center">DATOS SINTÉTICOS — no son Argentina</h1>

Se simula una EPH "de juguete" (columnas con los mismos nombres que la base individual de INDEC) con una verdad conocida (efecto de cohorte + perfil por edad + una pequeña oscilación de período), para comprobar que el pipeline recupera Ā. **Los números de esta sección no dicen nada sobre Argentina.**

In [ ]:
# ============ 2. DATASET SINTÉTICO + ASSERTS ============
banner("DATOS SINTÉTICOS — no son Argentina")

def f_edad_sintetica(a):  # perfil por edad inventado (logit)
    return 1.2 * np.exp(-((a - 40) / 18.0) ** 2) - 1.0

def alpha_sintetico(c):   # efecto de cohorte inventado (logit)
    return -0.2 + 0.008 * (np.asarray(c) - 1970)

def periodo_sintetico(y):
    return 0.10 * np.sin((np.asarray(y) - 2003) / 3.0)

def s65_sintetico(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)

def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    rng = np.random.default_rng(seed)
    filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(alpha_sintetico(y - a) + f_edad_sintetica(a) + periodo_sintetico(y))
                u = rng.random(n_por_edad)
                reg = u < p
                otro = ~reg
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))   # no registrados: 45% ocupados, resto inactivos
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)  # parte son cuentapropistas
                pp07h = np.where(cat == 3, np.where(reg, 1, 2), np.nan)
                pp07i = np.where(cat == 2, 2, np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad),
                                           "CH06": a, "ESTADO": estado, "CAT_OCUP": cat, "PP07H": pp07h,
                                           "PP07I": pp07i, "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)

df_sint = generar_eph_sintetica()
print("Filas sintéticas:", len(df_sint))
cohortes_sint = np.arange(1940, 2036)
s65_sint = pd.Series(s65_sintetico(cohortes_sint), index=cohortes_sint)
res_s = pipeline(df_sint, s65_sint, cohortes=cohortes_sint)
t = res_s["tabla"]

# verdad conocida
dens_true = pd.DataFrame(_sig(alpha_sintetico(cohortes_sint)[:, None] + f_edad_sintetica(EDADES)[None, :]),
                         index=cohortes_sint, columns=EDADES)
a_true, _ = abar_por_cohorte(dens_true, s65_sint)
t["A_bar_verdad_sintetica"] = a_true
t["error"] = t["A_bar"] - t["A_bar_verdad_sintetica"]

# --- asserts de control ---
chequeos(res_s)
obs = t[t["n_edades_obs"] >= MIN_EDADES_COHORTE]
mae = obs["error"].abs().mean()
print(f"[sintético] error absoluto medio de Ā en cohortes observadas: {mae:.3f} años")
assert mae < 1.5, "el pipeline no recupera la verdad sintética (tolerancia 1,5 años; el efecto de período sintético se confunde parcialmente con cohorte/edad: problema APC)"
fut = t[t["n_edades_obs"] < MIN_EDADES_COHORTE]
print(f"[sintético] error medio en cohortes SIN datos (extrapoladas con 'ultimas_k'): {fut['error'].mean():+.3f} años "
      "(sesgo esperado si hay tendencia entre cohortes: es el costo del supuesto de extrapolación)")
# monotonía en S65 a igual densidad (test directo)
a_lo, _ = abar_por_cohorte(res_s["dens"], s65_sint - 0.05)
a_hi, _ = abar_por_cohorte(res_s["dens"], s65_sint + 0.02)
assert np.all(a_hi.values > a_lo.values), "Ā debería crecer con S65"
# densidad cero => Ā cero ; densidad 1 => Ā = sum S(a) <= 47
z, _ = abar_por_cohorte(res_s["dens"] * 0.0, s65_sint); assert np.allclose(z, 0)
u, S1 = abar_por_cohorte(res_s["dens"] * 0.0 + 1.0, s65_sint); assert np.all(u <= 47) and np.all(u > 0)
print("Todos los asserts sintéticos OK.")

# gráficos
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for c, col in [(1950, "C0"), (1970, "C1"), (1990, "C2"), (2000, "C3")]:
    cc = res_s["cells"][res_s["cells"]["cohorte"] == c]
    ax[0].scatter(cc["edad"], cc["dens"], s=10, color=col, alpha=.6)
    ax[0].plot(EDADES, res_s["dens"].loc[c].values, color=col, label=f"cohorte {c}")
ax[0].set_xlabel("edad"); ax[0].set_ylabel("densidad d(a)"); ax[0].legend(); ax[0].set_title("SINTÉTICO: puntos observados vs modelo")
ax[1].plot(t.index, t["A_bar"], label="estimado"); ax[1].plot(t.index, t["A_bar_verdad_sintetica"], "--", label="verdad sintética")
ax[1].set_xlabel("cohorte"); ax[1].set_ylabel("Ā (años)"); ax[1].legend()
ax[1].set_title("SINTÉTICO: Ā por cohorte")
fig.suptitle("DATOS SINTÉTICOS — no son Argentina", color="red", fontsize=16, fontweight="bold")
plt.tight_layout(); plt.show()
banner("FIN PRUEBA SINTÉTICA — estos números NO son Argentina")

### 8.3 Prueba del lector de archivos (con un archivo de juguete en formato INDEC)

Escribe un subconjunto del dataset **sintético** en una carpeta temporal con el nombre `usu_individual_T100.txt` (separador `;`), y verifica que `find_eph_files` + `read_all` + `pipeline` dan lo mismo que con el DataFrame en memoria. No toca `datos/eph/`.

In [ ]:
banner("DATOS SINTÉTICOS — prueba del lector, no son Argentina")
with tempfile.TemporaryDirectory() as tmp:
    sub = df_sint[df_sint["ANO4"].isin([2010, 2011, 2012, 2013])].copy()
    sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]  # mezcla de mayúsculas/minúsculas
    sub.to_csv(os.path.join(tmp, "usu_individual_T100.txt"), sep=";", index=False)
    srcs = find_eph_files([tmp]); assert len(srcs) == 1
    leido = read_all(srcs)
    assert set(NEEDED) <= set(leido.columns)
    assert len(leido) == len(sub)
    # zip con el nombre típico de INDEC
    zp = os.path.join(tmp, "EPH_usu_1_Trim_2012_txt.zip")
    with zipfile.ZipFile(zp, "w") as z:
        z.write(os.path.join(tmp, "usu_individual_T100.txt"), "usu_individual_T112.txt")
    os.remove(os.path.join(tmp, "usu_individual_T100.txt"))
    srcs2 = find_eph_files([tmp]); assert len(srcs2) == 1 and srcs2[0]["member"] is not None
    assert len(read_all(srcs2)) == len(sub)
print("Lector OK (txt y zip) con datos sintéticos.")
# find_eph_files sobre una carpeta inexistente => lista vacía, sin error
assert find_eph_files(["/ruta/que/no/existe"]) == []

### 8.4 Corrida con datos REALES (solo si están en `datos/eph/`)

Si no hay archivos, el notebook termina limpio con "faltan datos". Si los hay, aplica el mismo `pipeline` y guarda `salidas_tbp/abar_eph_por_cohorte.csv` (carpeta ignorada por git). Las cifras de referencia 14,2 (ANSES, observado, sesgado a la baja) y 30 (requisito legal de referencia, no un máximo) se muestran solo como contexto; **no son objetivo de calibración**.

In [ ]:
# ============ 4. CORRIDA REAL ============
srcs_reales = find_eph_files(RUTAS_CANDIDATAS)
HAY_DATOS = len(srcs_reales) > 0

if not HAY_DATOS:
    print("=" * 78)
    print("FALTAN DATOS: no se encontraron bases individuales de la EPH en", RUTAS_CANDIDATAS)
    print("Qué hacer: bajar las bases individuales (usu_individual_T*.txt o EPH_usu_*_txt.zip) desde INDEC")
    print("y dejarlas en datos/eph/ (ver datos/eph/README.md). No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    s65_path = next((p for p in RUTAS_S65 if os.path.exists(p)), None)
    if s65_path is None:
        print("FALTAN DATOS: no se encontró tbp_argentina_cohortes_1950_2035.csv (S65 por cohorte).")
        HAY_DATOS = False
if HAY_DATOS:
    print(f"Archivos EPH encontrados: {len(srcs_reales)}")
    df_real = read_all(srcs_reales)
    print("Años cubiertos:", int(df_real["ANO4"].min()), "-", int(df_real["ANO4"].max()), "| filas:", len(df_real))
    s65_real = pd.read_csv(s65_path).set_index("cohorte")["S65"]
    coh_real = s65_real.index
    escenarios = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
                  "pp07i": dict(ind_rule="pp07i"),
                  "indep_50pct (supuesto)": dict(ind_rule="escenario", ind_share=0.5)}
    salida = {}
    for nombre, kw in escenarios.items():
        r = pipeline(df_real, s65_real, cohortes=coh_real, **kw)
        chequeos(r)
        salida[nombre] = r["tabla"]["A_bar"]
        salida["n_edades_obs"] = r["tabla"]["n_edades_obs"]
    out = pd.DataFrame(salida)
    os.makedirs("salidas_tbp", exist_ok=True)
    out.to_csv("salidas_tbp/abar_eph_por_cohorte.csv")
    print(out.loc[[c for c in (1960, 1970, 1980, 1990, 2000) if c in out.index]].round(2))
    print("Referencias de contexto (NO objetivo): ANSES observado 14,2 (sesgado a la baja); requisito legal 30 (techo).")

## 9. 🆕 Escenario propio de natalidad (RENAPER + DEIS + INDEC)

**Problema.** Para las cohortes 2025 en adelante, la serie principal usa los nacimientos proyectados por la ONU (WPP julio 2024): 508 mil en 2025, un **rebote artificial**. La ONU proyectó 2023 y 2024 *antes* de tener el dato: estimó 504 mil y 507 mil, y los nacimientos reales (DEIS) fueron **460.902** y **413.135**. Los datos recientes del RENAPER (2025 provisorio) y de la TGF indican que la caída continúa.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `DATA`, `OUT` en memoria):

1. arma un dato base 2025 **estimado** (RENAPER ajustado por la relación RENAPER/DEIS);
2. proyecta nacimientos 2025–2035 con tres escenarios de TGF **propuestos para que el grupo los valide**;
3. exporta `natalidad_escenarios_2025_2035.csv`;
4. recalcula $TBP_C$ y $TBP_A$ para las cohortes 2025–2035 y los compara con la serie actual (N de la ONU).

> **Aviso 1.** Los valores del RENAPER son los de la **descarga CSV del tablero** (nacimientos identificados y TGF, total país). El de **2025 es provisorio** y no equivale a los nacimientos *ocurridos* del DEIS.
> **Aviso 2.** $J$ (población 65+) **sigue siendo el de la ONU y NO está re-ponderado todavía**: eso se hace en la sección 10. Mientras tanto, el nivel de $TBP$ de las cohortes ≥2025 es parcial.

### 9.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: los datos del RENAPER y la población por edad simple de la ONU (mujeres, variante media). Rutas candidatas: `../datos/procesados`, `datos/procesados`, `DATA` y `datos_tbp`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_9 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_9 = ["../datos/procesados", "datos/procesados", DATA, "datos_tbp", "../datos_tbp"]

def buscar_archivo_9(nombre):
    for c in CANDS_9:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw)
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_9}/{nombre}", p)
    return p

f_ren = buscar_archivo_9("renaper_natalidad_2012_2025.csv")
f_pob = buscar_archivo_9("un_wpp2024_argentina_poblacion_edad_simple.csv")
print("RENAPER :", os.path.abspath(f_ren)); print("Poblacion ONU:", os.path.abspath(f_pob))

ren = pd.read_csv(f_ren).set_index("anio")
ren["nac"] = ren["nacimientos_identificados_miles"] * 1000      # personas
pob = pd.read_csv(f_pob, low_memory=False)
if "Variant" in pob.columns:
    pob = pob[pob["Variant"] == "Medium"]
pob["edad"] = pd.to_numeric(pob["AgeGrpStart"], errors="coerce")
# M_t: mujeres de 15 a 49 anios en t (ONU medio), en miles
M = pob[(pob.edad >= 15) & (pob.edad <= 49)].groupby("Time")["PopFemale"].sum()

deis = s["deis_nacimientos"]; onu = s["un_births_miles"] * 1000
display(pd.DataFrame({"DEIS": deis.loc[2019:2024], "RENAPER": ren.nac.loc[2019:2024],
                      "ONU": onu.loc[2019:2024]}).round(0))
print("TGF RENAPER:", ren.tgf_hijos_por_mujer.loc[2022:2025].to_dict())
print("Mujeres 15-49 (miles, ONU medio):", M.loc[[2025, 2030, 2035]].round(0).to_dict())

### 9.2 Dato base 2025: ESTIMACIÓN a partir de un dato provisorio

Los nacimientos del RENAPER son *personas identificadas* y no coinciden exactamente con los *nacimientos ocurridos* (DEIS). Para llevar el dato RENAPER a la escala DEIS (la que usa la serie hasta 2024) se lo divide por la **relación RENAPER/DEIS promedio 2022–2024**:

$$N_{2025}^{est} = \frac{N_{2025}^{RENAPER}}{\overline{\left(N^{RENAPER}/N^{DEIS}\right)}_{2022\text{–}2024}}$$

**Por qué esa ventana:** la relación es estable (0,98–1,05 en 2012–2024) y los años recientes son los más comparables con 2025 (misma cobertura de DNI digital). Es una **ESTIMACIÓN**, **no un dato oficial**: combina un valor provisorio con una lectura a ojo de un gráfico.

In [ ]:
# ---- Parametros del dato base (modificables) ----
VENTANA_RATIO = (2022, 2024)        # anios usados para la relacion RENAPER/DEIS
ANIO_BASE = 2025

ratio = (ren.nac / deis).loc[2012:2024]
r_med = ratio.loc[VENTANA_RATIO[0]:VENTANA_RATIO[1]].mean()
N_REN_2025 = ren.nac[ANIO_BASE]
N_2025_EST = N_REN_2025 / r_med
TGF_2025_OBS = ren.tgf_hijos_por_mujer[ANIO_BASE]

print("Relacion RENAPER/DEIS 2012-2024:"); print(ratio.round(3).to_string())
print(f"\nMedia {VENTANA_RATIO[0]}-{VENTANA_RATIO[1]}: {r_med:.3f}")
print(f"RENAPER 2025 (provisorio): {N_REN_2025:,.0f}")
print(f"N_2025 ESTIMADO = {N_2025_EST:,.0f}  <- ESTIMACION a partir de un dato provisorio, NO dato oficial")
print(f"(ONU proyecta {onu[2025]:,.0f} para 2025: {onu[2025]/N_2025_EST - 1:+.0%} respecto de la estimacion)")

### 9.3 Escenarios 2025–2035 (propuesta a validar por el grupo)

**Método.** Los nacimientos de un año son (TGF × mujeres en edad fértil) por una constante que convierte "TGF" a "nacimientos":

$$N_t = TGF_t \times K \times \frac{M_t}{M_{2025}},\qquad K = \frac{N_{2025}^{est}}{TGF_{2025}}$$

* $M_t$ = mujeres de 15–49 años en $t$ (ONU medio, `PopFemale`). Capta el *efecto de composición* (cuántas mujeres hay en edad de tener hijos): crece ~3% entre 2025 y 2035.
* $K$ calibra el nivel en 2025, de modo que los tres escenarios **coinciden en 2025** y difieren solo por la trayectoria de la TGF. Es una aproximación: supone que la estructura de edades de la fecundidad no cambia (la TGF real usa tasas por edad).

**Los tres escenarios** (parámetros editables en la celda siguiente):

* **(a) Persistencia:** la TGF se queda en 1,04 (nivel observado 2025). Referencia "nada cambia".
* **(b) Recuperación lenta:** sube linealmente de 1,04 (2025) a 1,32 (2035), que es el valor de la **variante media de INDEC para 2035** (INDEC, proyecciones 2022–2040, pp. 26–30). Es el escenario "optimista".
* **(c) Caída adicional:** baja a 0,95 en 2028 y se estabiliza. Es una cota inferior ilustrativa, no una predicción.

**Contexto INDEC.** Las proyecciones de INDEC usaron nacimientos DEIS solo hasta 2023 y suponen una TGF media de 1,27 (2025), 1,24 (2030) y 1,32 (2035). La TGF observada 2025 del RENAPER (~1,04) queda **por debajo incluso del mínimo de la variante baja de INDEC (1,17)**. Por eso INDEC aparece solo como TGF de referencia (no se calculan nacimientos con la TGF INDEC: su denominador de mujeres y su calibración son otros).

In [ ]:
# ============ PARAMETROS DE LOS ESCENARIOS (modificables) ============
TGF_2025 = TGF_2025_OBS         # 1,04: TGF RENAPER 2025 (provisorio)
TGF_REC_2035 = 1.32             # (b) valor INDEC variante media 2035
TGF_CAIDA_MIN = 0.95            # (c) piso de la caida adicional
ANIO_CAIDA = 2028               # (c) anio en que se alcanza el piso
ANIOS = list(range(2025, 2036))
A_LEGAL_9, A_OBS_9 = 30.0, 14.2  # 30 = requisito legal de referencia (NO es un maximo); 14,2 = observado ANSES
# Referencia INDEC (proyecciones 2022-2040, pp. 26-30, ya verificado)
INDEC_MEDIA = {2025: 1.27, 2030: 1.24, 2035: 1.32, 2040: 1.40}
INDEC_MIN_BAJA = {2025: 1.28, 2030: 1.22, 2035: 1.17}     # minimos citados de la variante baja
# =====================================================================

K = N_2025_EST / TGF_2025

def tgf_escenario(nombre, t):
    if nombre == "persistencia":
        return TGF_2025
    if nombre == "recuperacion_lenta":
        return TGF_2025 + (TGF_REC_2035 - TGF_2025) * (t - ANIOS[0]) / (ANIOS[-1] - ANIOS[0])
    if nombre == "caida_adicional":
        return TGF_2025 + (TGF_CAIDA_MIN - TGF_2025) * min(t - ANIOS[0], ANIO_CAIDA - ANIOS[0]) / (ANIO_CAIDA - ANIOS[0])
    raise ValueError(nombre)

ESCENARIOS = ["persistencia", "recuperacion_lenta", "caida_adicional"]
filas = []
for e in ESCENARIOS:
    for t in ANIOS:
        tg = tgf_escenario(e, t)
        filas.append(dict(anio=t, escenario=e, tgf=tg, nacimientos=tg * K * M[t] / M[ANIO_BASE]))
# referencias: ONU (nacimientos) e INDEC media (solo TGF, interpolada linealmente entre anios publicados)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="onu_medio", tgf=np.nan, nacimientos=onu[t]))
ks = sorted(INDEC_MEDIA)
for t in ANIOS:
    filas.append(dict(anio=t, escenario="indec_media_tgf", tgf=np.interp(t, ks, [INDEC_MEDIA[k] for k in ks]), nacimientos=np.nan))
esc = pd.DataFrame(filas)

piv = esc.pivot(index="anio", columns="escenario", values="nacimientos")[ESCENARIOS + ["onu_medio"]]
display((piv / 1000).round(1).loc[[2025, 2026, 2028, 2030, 2035]].rename_axis("miles de nacimientos"))
print(f"TGF observada 2025 = {TGF_2025:.2f} < minimo variante baja INDEC 2035 = {INDEC_MIN_BAJA[2035]:.2f} y 2025 = {INDEC_MIN_BAJA[2025]:.2f}")

### 9.4 Salida: `natalidad_escenarios_2025_2035.csv`

Formato largo: `anio, escenario, tgf, nacimientos`. Incluye los tres escenarios y dos referencias: `onu_medio` (solo nacimientos; la TGF de la ONU no se cargó acá) e `indec_media_tgf` (solo TGF, interpolada linealmente entre 2025, 2030 y 2035; **nacimientos vacíos a propósito**: no se tiene la serie de nacimientos de INDEC y calcularla con otra metodología mezclaría supuestos).

In [ ]:
DEST = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST, exist_ok=True)
f_out = os.path.join(DEST, "natalidad_escenarios_2025_2035.csv")
esc.round({"tgf": 4, "nacimientos": 0}).to_csv(f_out, index=False)
print("Escrito:", os.path.abspath(f_out), esc.shape)

### 9.5 Recalcular el TBP con cada escenario

Se reemplaza $N_t$ por el del escenario **solo para las cohortes ≥2025** (cohortes ≤2024 quedan idénticas: DEIS). El resto de los insumos es el de `d`: $S_{65}$, $E_{65}$ y $J$.

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

> **J NO está re-ponderado todavía** (sigue siendo la población 65+ de la ONU, que descansa en los nacimientos ONU anteriores). Con menos nacimientos, la población 65+ de 2090–2100 que proyecta la ONU también sería menor, así que el denominador está sobreestimado y el efecto aquí es una **cota parcial**. La corrección va en la sección 10.

In [ ]:
base9 = d.set_index("cohorte")[["N_t", "S65", "J", "E65", "TBP_C_legal", "TBP_A_legal", "TBP_C_obs", "TBP_A_obs"]].copy()
N_snapshot = base9["N_t"].copy()                      # para el control "no cambia <=2024"

def serie_N(nombre):
    # N por cohorte: DEIS hasta 2024 (igual que d); desde 2025 el escenario (o la ONU, serie actual)
    N = base9["N_t"].copy()
    if nombre != "actual_onu":
        N.loc[ANIOS] = esc[esc.escenario == nombre].set_index("anio")["nacimientos"].loc[ANIOS].values
    return N

def tbp_9(N, A, tipo):
    c = N * base9.S65 * A * TAU / (base9.J * RHO)
    return c if tipo == "C" else c / base9.E65

NOMBRES = ["actual_onu"] + ESCENARIOS
N9 = {n: serie_N(n) for n in NOMBRES}
T9 = {(n, A, tipo): tbp_9(N9[n], A, tipo) for n in NOMBRES for A in (A_LEGAL_9, A_OBS_9) for tipo in "CA"}

# la serie "actual" reproduce la de d (control de consistencia)
assert np.allclose(T9[("actual_onu", A_LEGAL_9, "C")], base9.TBP_C_legal)
assert np.allclose(T9[("actual_onu", A_OBS_9, "A")], base9.TBP_A_obs)
print("OK: la serie 'actual_onu' recalculada reproduce TBP_C/TBP_A de d")

### 9.6 Gráficos

(i) Nacimientos 2012–2035: DEIS, RENAPER (sin ajustar), ONU y los tres escenarios (el punto 2025 común es la **estimación**). (ii) $TBP_C$ por cohorte 2000–2035 para $\bar A=30$ y $\bar A=14{,}2$, con los tres escenarios y la serie actual; la línea vertical marca el corte 2024/2025. **$\bar A=30$ es el requisito legal de referencia, no un máximo.** En ambos paneles de $TBP_C$, $J$ es el de la ONU (no re-ponderado).

In [ ]:
COL9 = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7", "actual_onu": "#444444"}
ETQ9 = {"persistencia": "(a) Persistencia (TGF 1,04)", "recuperacion_lenta": "(b) Recuperación lenta (a 1,32)",
        "caida_adicional": "(c) Caída adicional (a 0,95)", "actual_onu": "Serie actual (N de la ONU)"}

fig, ax = plt.subplots(1, 3, figsize=(20, 5.4))
a = ax[0]
a.plot(deis.loc[2012:2024].index, deis.loc[2012:2024] / 1e3, color="#0072b2", lw=2, label="DEIS (observado)")
a.plot(ren.index, ren.nac / 1e3, "s--", color="#e69f00", ms=4, lw=1.2, label="RENAPER (2025 prov.)")
a.plot(onu.loc[2012:2035].index, onu.loc[2012:2035] / 1e3, color="#444444", ls=":", lw=2, label="ONU WPP 2024 (medio)")
for e in ESCENARIOS:
    p = piv[e]; a.plot(p.index, p / 1e3, color=COL9[e], lw=2, label=ETQ9[e])
a.plot([2025], [N_2025_EST / 1e3], "o", color="k", ms=6, label=f"N 2025 estimado ({N_2025_EST/1e3:.0f} mil)")
a.axvline(2024.5, color="gray", ls="--", lw=1); a.set_title("(i) Nacimientos 2012–2035 (miles)")
a.set_xlabel("año"); a.grid(alpha=.3); a.legend(fontsize=7.5)

for a, AA, tit in [(ax[1], A_LEGAL_9, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                   (ax[2], A_OBS_9, "Ā = 14,2 (observado ANSES, sesgado a la baja)")]:
    for n in NOMBRES:
        x = T9[(n, AA, "C")].loc[2000:2035]
        a.plot(x.index, x, color=COL9[n], lw=2.4 if n == "actual_onu" else 1.8, ls=":" if n == "actual_onu" else "-", label=ETQ9[n])
    a.axvline(2024.5, color="gray", ls="--", lw=1)
    a.text(2024.8, a.get_ylim()[1] * 0.98, "2024/2025", fontsize=8, color="gray", va="top")
    a.set_title(f"(ii) TBP_C por cohorte, {tit}", fontsize=9.5); a.set_ylabel("TBP_C (años)")
    a.set_xlabel("año de nacimiento de la cohorte"); a.grid(alpha=.3); a.legend(fontsize=7.5, loc="upper right")
fig.suptitle("Escenarios propios de natalidad 2025–2035 — J todavía NO re-ponderado (sección 10)", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95])
os.makedirs(OUT, exist_ok=True)
plt.savefig(f"{OUT}/seccion_09_natalidad.png", dpi=130); plt.show()

### 9.7 Tabla resumen: cohortes 2025, 2030 y 2035

Nacimientos y $TBP$ por escenario frente a la serie actual (N de la ONU). Columna `Δ vs actual` = variación porcentual del $TBP_C$ respecto de la serie actual. **$J$ de la ONU, no re-ponderado.**

In [ ]:
COHS = [2025, 2030, 2035]
res = []
for n in NOMBRES:
    for c in COHS:
        r = dict(cohorte=c, escenario=n, N=N9[n][c])
        for A, lab in [(A_LEGAL_9, "A30"), (A_OBS_9, "A14,2")]:
            r[f"TBP_C ({lab})"] = T9[(n, A, "C")][c]
            r[f"TBP_A ({lab})"] = T9[(n, A, "A")][c]
        r["Δ vs actual (%)"] = 100 * (T9[(n, A_LEGAL_9, "C")][c] / T9[("actual_onu", A_LEGAL_9, "C")][c] - 1)
        res.append(r)
resumen = pd.DataFrame(res).sort_values(["cohorte", "escenario"]).set_index(["cohorte", "escenario"])
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
display(resumen.round({"N": 0, "Δ vs actual (%)": 1}))
print("Nota: J = poblacion 65+ de la ONU (NO re-ponderado; se hace en la seccion 10). Resultados parciales.")

### 9.8 Controles (asserts)

1. Relación RENAPER/DEIS en [0,95; 1,06] para 2012–2024 (se excluye 2020, año de pandemia).
2. Nacimientos de todos los escenarios positivos.
3. Los tres escenarios coinciden en 2025 (y valen $N_{2025}^{est}$).
4. El ajuste no cambia las cohortes ≤2024 (ni `N_t` ni el TBP).

In [ ]:
r_chk = ratio.drop(2020)
assert r_chk.between(0.95, 1.06).all(), r_chk[~r_chk.between(0.95, 1.06)]
print(f"OK 1) RENAPER/DEIS en [0,95; 1,06]: min {r_chk.min():.3f} | max {r_chk.max():.3f} (2012-2024 sin 2020; 2020 = {ratio[2020]:.3f})")

for e in ESCENARIOS:
    assert (esc[esc.escenario == e].nacimientos > 0).all()
print("OK 2) nacimientos > 0 en los 3 escenarios, 2025-2035")

n25 = esc[(esc.anio == 2025) & esc.escenario.isin(ESCENARIOS)].nacimientos
assert np.allclose(n25, N_2025_EST) and np.ptp(n25) < 1e-6
print(f"OK 3) los 3 escenarios coinciden en 2025: {n25.iloc[0]:,.0f}")

ant = base9.index <= 2024
for n in ESCENARIOS:
    assert np.array_equal(N9[n][ant].values, N_snapshot[ant].values)
    assert np.allclose(T9[(n, A_LEGAL_9, "C")][ant], base9.TBP_C_legal[ant])
    assert np.allclose(T9[(n, A_OBS_9, "A")][ant], base9.TBP_A_obs[ant])
assert np.array_equal(d.set_index("cohorte")["N_t"].values, N_snapshot.values)   # d no fue modificado
assert esc.anio.min() == 2025
print("OK 4) cohortes <=2024 sin cambios (N_t y TBP) y `d` intacto")

### 9.9 Lectura y límites

* **Qué se puede decir:** la serie actual (N de la ONU) **sobrestima** los nacimientos de las cohortes 2025+ respecto de lo que muestran RENAPER y la TGF observada; incluso con una recuperación lenta hasta el valor INDEC 2035, las cohortes 2025–2030 quedan por debajo de lo que supone la ONU. Con persistencia o caída, $TBP_C$ de las cohortes recientes cae en forma sostenida frente a la serie actual.
* **Qué NO se puede decir:** que estos escenarios sean pronósticos; que 390 mil sea el dato oficial 2025 (es una estimación sobre un dato provisorio); que el nivel absoluto del $TBP$ de las cohortes ≥2025 sea definitivo (**$J$ no está re-ponderado**: sección 10).
* **Supuestos a validar por el grupo:** (i) estructura de edades de la fecundidad constante (la proporcionalidad TGF × mujeres 15–49); (ii) ventana 2022–2024 para la relación RENAPER/DEIS; (iii) los tres valores de TGF (1,04 / 1,32 / 0,95 en 2028).
* **Pendiente:** reemplazar el 2025 provisorio por el definitivo cuando el RENAPER lo publique.

## 10. 🆕 Consistencia de J: reponderación con nacimientos propios

**El problema (circularidad).** $J_{t+65}$ es la población de 65+ años que la ONU proyecta para el año $t+65$. Esa población está construida con los **nacimientos de la ONU** de las cohortes que la componen (nacidas entre $t-35$ y $t$). Pero en $N_t$ ya no usamos los nacimientos de la ONU: usamos DEIS (observado, hasta 2024) y, desde 2025, los escenarios propios de la sección 9. Es decir, el numerador y el denominador del $TBP$ cuentan **historias de natalidad distintas**: $N$ dice "nacieron pocos", $J$ supone "nacieron los que dijo la ONU". Si la natalidad cae, la población 65+ del futuro también tiene que ser menor, y el denominador está sobreestimado.

**Qué hace esta sección** (fragmento autónomo; depende solo de `d`, `s`, `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA`, `OUT` en memoria y de los archivos que carga él mismo):

1. reconstruye $J$ como suma por edad de la población ONU y la **repondera por cohorte** con $w(c)=N_{propio}(c)/N_{ONU}(c)$;
2. compara $J$ original vs reponderado, para cada escenario;
3. recalcula $TBP_C$ y $TBP_A$ en tres versiones: (i) serie actual, (ii) $N$ propio con $J$ ONU (inconsistente), (iii) $N$ propio con $J$ reponderado (consistente);
4. exporta `tbp_cohortes_J_reponderado.csv` y verifica con asserts.

### 10.1 Carga de datos

Se cargan dos archivos de `datos/procesados/`: la población ONU por edad simple (`PopTotal`, variante media, miles) y los escenarios de natalidad de la sección 9. Rutas candidatas: `../datos/procesados`, `datos/procesados` y `DATA`; si no están, se descargan del repositorio (URL *raw*) a la carpeta de salidas. Los nacimientos ONU y DEIS salen de `s`.

In [ ]:
import os, urllib.request, numpy as np, pandas as pd, matplotlib.pyplot as plt
try: display
except NameError: display = print

RAW_BASE_10 = "https://raw.githubusercontent.com/carolinamsfelipe/Tp-Indicadores/main/datos/procesados"
CANDS_10 = ["../datos/procesados", "datos/procesados", DATA]

def buscar_archivo_10(nombre):
    for c in CANDS_10:
        p = os.path.join(c, nombre)
        if os.path.exists(p):
            return p
    os.makedirs(OUT, exist_ok=True)          # fallback: descarga desde GitHub (raw) a la carpeta de salidas
    p = os.path.join(OUT, nombre)
    urllib.request.urlretrieve(f"{RAW_BASE_10}/{nombre}", p)
    return p

f_pob10 = buscar_archivo_10("un_wpp2024_argentina_poblacion_edad_simple.csv")
f_esc10 = buscar_archivo_10("natalidad_escenarios_2025_2035.csv")
print("Poblacion ONU :", os.path.abspath(f_pob10)); print("Escenarios    :", os.path.abspath(f_esc10))

pop10 = pd.read_csv(f_pob10, low_memory=False)
if "Variant" in pop10.columns:
    pop10 = pop10[pop10["Variant"] == "Medium"]
# PopTotal por edad simple (columnas 0..100; 100 = grupo abierto 100+) y anio, en PERSONAS
P = pop10.pivot_table(index="Time", columns="AgeGrpStart", values="PopTotal") * 1000
P.columns = P.columns.astype(int)

esc10 = pd.read_csv(f_esc10)
ESC10 = ["persistencia", "recuperacion_lenta", "caida_adicional"]      # escenarios propios (seccion 9)
assert set(ESC10 + ["onu_medio"]) <= set(esc10.escenario.unique()), esc10.escenario.unique()
NAC10 = esc10.pivot(index="anio", columns="escenario", values="nacimientos")[ESC10 + ["onu_medio"]]

N_onu10 = s["un_births_miles"] * 1000          # nacimientos ONU (personas), 1950-2100
N_deis10 = s["deis_nacimientos"]               # nacimientos DEIS observados, 1914-2024
print("Edades en la tabla ONU:", P.columns.min(), "a", P.columns.max(), "| anios:", P.index.min(), "a", P.index.max())
print("Escenarios cargados:", ESC10, "+ onu_medio | anios", NAC10.index.min(), "a", NAC10.index.max())
display((NAC10 / 1000).round(1).loc[[2025, 2030, 2035]].rename_axis("miles de nacimientos"))

### 10.2 Método: reponderación por cohorte

La población de $a$ años en el año calendario $y$ pertenece a la cohorte $c=y-a$. Si esa cohorte tuvo, según nuestros datos, una fracción $w(c)$ de los nacimientos que supuso la ONU, asumimos que **le quedan $w(c)$ veces los sobrevivientes** que proyecta la ONU. Entonces:

$$J^{rw}(y)=\sum_{a=65}^{100+} \text{PopTotal}_{ONU}(a,y)\cdot w(y-a),\qquad w(c)=\frac{N_{propio}(c)}{N_{ONU}(c)}$$

con $N_{propio}$ = DEIS para $c\le 2024$ y el escenario para $c\ge 2025$. Se aplica a $y=t+65$ para $t=1950,\dots,2035$.

**Por qué esta forma (didáctica).** Es una regla de tres por cohorte: si una generación fue 20% más chica de lo que suponía la ONU, la población de esa generación a los 70 años también lo es. Se corrige solo lo que la circularidad contamina (los nacimientos) y se deja intacto el resto de la proyección ONU (mortalidad, migración).

**Convenciones (documentadas):**

* $w=1$ para cohortes $c<1950$: la serie `un_births_miles` empieza en 1950, no hay nacimientos ONU contra los cuales comparar.
* $w=1$ para 1971–1974: la serie DEIS tiene un hueco y el cuaderno principal completa $N_t$ con la ONU (`fuente_N`); se usa la misma regla.
* El grupo abierto **100+** es una sola fila de la tabla ONU; se le asigna $w(y-100)$ (cohorte más joven del grupo, la más numerosa). Pesa muy poco en $J$.
* La población ONU es de **mitad de año**, así que "edad $a$ en $y$" mezcla las cohortes $y-a$ e $y-a-1$. Se usa $c=y-a$ (la misma convención del cuaderno principal, donde la cohorte $t$ cumple 65 en $t+65$) y se chequea la robustez promediando ambas.

**Supuestos y límites.** (a) Mortalidad y migración de cada cohorte proporcionales a los de la proyección ONU (se reescala el tamaño, no el comportamiento). (b) Para $c\le 2024$, $w=$ DEIS/ONU mezcla la corrección por la circularidad con la **diferencia de nivel entre registro (DEIS) y estimación ONU**, que existe también en cohortes viejas (media DEIS/ONU 1950–2024: 0,97). Por eso se muestra también una variante que ajusta solo 2023–2024.

In [ ]:
AGES10 = np.arange(65, 101)            # 65..99 edades simples + 100 (= grupo abierto 100+)

def pesos_cohorte(escenario=None, solo_2023_24=False):
    """w(c) = N_propio(c) / N_ONU(c).  w=1 para c<1950 (no hay nacimientos ONU en la serie).
    N_propio = DEIS para 1950<=c<=2024 y el escenario para 2025<=c<=2035.
    Cohortes 1971-1974: la serie DEIS tiene un hueco; el cuaderno principal las completa con la ONU (fuente_N),
    asi que alli w = 1 (misma regla que N_t de d).
    escenario=None -> w = 1 en TODAS las cohortes (control: debe reproducir J de la ONU).
    solo_2023_24=True -> variante de referencia: w=1 salvo 2023 y 2024 (DEIS) y los escenarios (>=2025)."""
    w = pd.Series(1.0, index=np.arange(1900, 2101))
    if escenario is None:
        return w
    for c in range(1950, 2025):
        if ((not solo_2023_24) or c >= 2023) and not np.isnan(N_deis10.get(c, np.nan)):
            w[c] = N_deis10[c] / N_onu10[c]
    for c in range(2025, 2036):
        w[c] = NAC10.loc[c, escenario] / N_onu10[c]
    assert w.notna().all()
    return w

def J_reponderado(w, anios, mezcla=0.0):
    """J_rw(y) = sum_{a=65}^{100} PopTotal_ONU(a, y) * w(y - a).
    Convencion para el grupo abierto 100+: se le asigna w(y - 100) (la cohorte mas joven del grupo; es
    la mas numerosa y, por tamanio del grupo, el error es de segundo orden).
    mezcla=0 -> cohorte = y - a (convencion del cuaderno principal: cohorte t llega a 65 en t+65).
    mezcla=0.5 -> promedia las cohortes y-a e y-a-1 (control de robustez: la poblacion ONU es de mitad de anio)."""
    out = {}
    for y in anios:
        c = y - AGES10
        wy = (1 - mezcla) * w.reindex(c).values + mezcla * w.reindex(c - 1).values
        out[y] = float((P.loc[y, AGES10].values * wy).sum())
    return pd.Series(out)

dd = d.set_index("cohorte")
ANIOS65 = [int(c) + 65 for c in dd.index]                  # 2015..2100
J_onu10 = pd.Series(dd.J.values, index=dd.index)         # J de la ONU, indexado por cohorte t (anio t+65)

def J_rw_coh(escenario, **kw):
    r = J_reponderado(pesos_cohorte(escenario, kw.pop("solo_2023_24", False)), ANIOS65, **kw)
    r.index = dd.index
    return r

JRW = {e: J_rw_coh(e) for e in ESC10 + ["onu_medio"]}      # onu_medio = DEIS<=2024 y ONU despues
print("Pesos w(c) = N_propio/N_ONU en cohortes recientes:")
display(pd.DataFrame({"w DEIS/ONU": pesos_cohorte("persistencia").loc[[2000, 2010, 2020, 2022, 2023, 2024]],
                      }).T.round(3))
display(pd.DataFrame({e: pesos_cohorte(e).loc[2025:2035:5] for e in ESC10}).T.round(3).rename_axis("w(c), c = 2025/2030/2035"))

### 10.3 J reponderado vs J ONU

Diferencia porcentual $J^{rw}/J^{ONU}-1$. Para $t\le 2024$ todos los escenarios dan lo mismo (el stock 65+ de $t+65$ solo incluye cohortes $\le t$). Los escenarios solo afectan a las cohortes $t\ge 2025$.

In [ ]:
# Diferencia porcentual J_rw vs J_ONU, cohortes t <= 2024 (stock 2015-2089)
def difpct(Jrw, rango):
    x = 100 * (Jrw / J_onu10 - 1).loc[rango[0]:rango[1]]
    return x

rows = []
for nombre, kw in [("Método principal (DEIS 1950-2024)", {}),
                   ("Variante: solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("Robustez: cohorte = promedio de y-a e y-a-1", {"mezcla": 0.5})]:
    x = difpct(J_rw_coh("persistencia", **kw), (1950, 2024))
    rows.append({"variante": nombre, "media (%)": x.mean(), "media |Δ| (%)": x.abs().mean(),
                 "máx |Δ| (%)": x.abs().max(), "cohorte del máx": int(x.abs().idxmax())})
tab_dif = pd.DataFrame(rows).set_index("variante")
print(tab_dif.round(2).to_string())
x_main = difpct(JRW["persistencia"], (1950, 2024))
print(f"t <= 2024, método principal: ΔJ media = {x_main.mean():+.2f}% | media del valor absoluto = {x_main.abs().mean():.2f}% | "
      f"máx |Δ| = {x_main.abs().max():.2f}% (cohorte {int(x_main.abs().idxmax())}, J ONU {J_onu10[int(x_main.abs().idxmax())]/1e6:.2f} M -> "
      f"{JRW['persistencia'][int(x_main.abs().idxmax())]/1e6:.2f} M)")
print("Cohortes 2020-2024 (ΔJ %):", x_main.loc[2020:2024].round(2).to_dict())
post = pd.DataFrame({e: 100 * (JRW[e] / J_onu10 - 1) for e in ESC10 + ["onu_medio"]}).loc[[2025, 2030, 2035]]
print("\nΔJ % para cohortes >= 2025 (J de la cohorte t = poblacion 65+ en t+65):")
display(post.round(2))

### 10.4 Recalcular el TBP en tres versiones

$$TBP_C=\frac{N(1-\mu)\bar A\tau}{J\rho},\qquad TBP_A=\frac{TBP_C}{E_r}$$

* **(i) serie actual:** $N$ = DEIS hasta 2024 y ONU después; $J$ ONU.
* **(ii) inconsistente:** $N$ de la sección 9; $J$ ONU sin reponderar.
* **(iii) consistente:** $N$ de la sección 9; $J$ reponderado con el mismo escenario.

Se usan $\bar A=30$ (requisito legal de referencia, **no un máximo**) y $\bar A=14{,}2$ (observado ANSES). $S_{65}$, $E_{65}$, $\tau$ y $\rho$ son los de `d`.

In [ ]:
A_LIST = [("A30", A_LEGAL), ("A14_2", A_OBS)]

def N_escenario(e):
    N = dd.N_t.copy()                                     # DEIS <= 2024 ; ONU >= 2025 (serie actual)
    N.loc[2025:2035] = NAC10.loc[2025:2035, e].values
    return N

def tbp(N, J, A, tipo):
    c = N * dd.S65 * A * TAU / (J * RHO)                 # TBP_C (años)
    return c if tipo == "C" else c / dd.E65             # TBP_A = TBP_C / E_r

# (i) serie actual: N ONU post-2024, J ONU.  (ii) N seccion 9, J ONU sin reponderar (inconsistente).  (iii) N seccion 9, J reponderado.
V = {}
for A_lab, A in A_LIST:
    for tp in "CA":
        V[("i", None, A_lab, tp)] = tbp(dd.N_t, dd.J, A, tp)
        for e in ESC10:
            V[("ii", e, A_lab, tp)]  = tbp(N_escenario(e), dd.J, A, tp)
            V[("iii", e, A_lab, tp)] = tbp(N_escenario(e), JRW[e], A, tp)

# la serie (i) reproduce d
assert np.allclose(V[("i", None, "A30", "C")], dd.TBP_C_legal) and np.allclose(V[("i", None, "A14_2", "A")], dd.TBP_A_obs)
print("OK: (i) reproduce TBP_C / TBP_A de d (Ā = 30 y 14,2)")

### 10.5 Gráficos

(a) $J$ original vs reponderado en año calendario ($t+65$, 2015–2100) y su diferencia porcentual. (b) $TBP_C$ por cohorte 2000–2035 con (i), (ii) y (iii), para cada escenario y para $\bar A=30$ y $14{,}2$. La línea vertical marca el corte 2024/2025.

In [ ]:
COL10 = {"i": "#444444", "ii": "#d55e00", "iii": "#0072b2"}
ETQ_E = {"persistencia": "Persistencia (TGF 1,04)", "recuperacion_lenta": "Recuperación lenta (a 1,32)",
         "caida_adicional": "Caída adicional (a 0,95)"}
COLE = {"persistencia": "#d55e00", "recuperacion_lenta": "#009e73", "caida_adicional": "#cc79a7"}

# ---- Figura 1: J original vs reponderado (anio calendario 2015-2100) ----
anio = np.array(ANIOS65)
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
ax[0].plot(anio, J_onu10.values / 1e6, color="#444444", lw=2.4, ls=":", label="J ONU (original)")
for e in ESC10:
    ax[0].plot(anio, JRW[e].values / 1e6, color=COLE[e], lw=1.8, label=f"J reponderado: {ETQ_E[e]}")
ax[0].set_title("Población 65+ (millones), año calendario t+65"); ax[0].set_xlabel("año calendario (t+65)")
ax[0].set_ylabel("millones"); ax[0].grid(alpha=.3); ax[0].legend(fontsize=8)
ax[0].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[0].text(2090.1, ax[0].get_ylim()[0] + 0.5, "cohortes\n≥2025", fontsize=7.5, color="gray")
for e in ESC10 + ["onu_medio"]:
    ax[1].plot(anio, 100 * (JRW[e] / J_onu10 - 1).values, color=COLE.get(e, "#444444"), lw=1.8,
               ls=":" if e == "onu_medio" else "-", label="Solo DEIS ≤2024 (ONU después)" if e == "onu_medio" else ETQ_E[e])
ax[1].axhline(0, color="gray", lw=.8); ax[1].axvline(2024 + 65 + .5, color="gray", ls="--", lw=1)
ax[1].set_title("J reponderado vs J ONU (diferencia %)"); ax[1].set_xlabel("año calendario (t+65)")
ax[1].set_ylabel("%"); ax[1].grid(alpha=.3); ax[1].legend(fontsize=8)
fig.suptitle("Sección 10: población 65+ original (ONU) vs reponderada con nacimientos propios", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.94]); plt.savefig(f"{OUT}/seccion_10_J_reponderado.png", dpi=130); plt.show()

# ---- Figura 2: TBP_C por cohorte 2000-2035, (i), (ii), (iii) ----
fig, ax = plt.subplots(2, 3, figsize=(19, 9), sharex=True)
for i, (A_lab, A, tit) in enumerate([("A30", A_LEGAL, "Ā = 30 (requisito legal de referencia, no un máximo)"),
                                     ("A14_2", A_OBS, "Ā = 14,2 (observado ANSES)")]):
    for j, e in enumerate(ESC10):
        a = ax[i, j]
        a.plot(V[("i", None, A_lab, "C")].loc[2000:2035], color=COL10["i"], lw=2.2, ls=":", label="(i) serie actual (N ONU, J ONU)")
        a.plot(V[("ii", e, A_lab, "C")].loc[2000:2035], color=COL10["ii"], lw=1.8, label="(ii) N propio, J ONU (inconsistente)")
        a.plot(V[("iii", e, A_lab, "C")].loc[2000:2035], color=COL10["iii"], lw=2.2, label="(iii) N propio, J reponderado (consistente)")
        a.axvline(2024.5, color="gray", ls="--", lw=1)
        a.set_title(f"{ETQ_E[e]}\nTBP_C, {tit}", fontsize=9); a.grid(alpha=.3)
        if j == 0: a.set_ylabel("TBP_C (años)")
        if i == 1: a.set_xlabel("año de nacimiento de la cohorte")
        a.legend(fontsize=7.5, loc="lower left")
fig.suptitle("TBP_C por cohorte 2000–2035: serie actual vs N propio con J sin reponderar vs J reponderado", fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.savefig(f"{OUT}/seccion_10_TBP_C_reponderado.png", dpi=130); plt.show()

### 10.6 Tabla: cohortes 2024, 2030 y 2035

La columna `TBP_C(A30) vs (i) %` compara con la serie actual. Debajo, el efecto aislado de reponderar $J$ ((iii) vs (ii), mismo $N$) y la sensibilidad a la convención de pesos.

In [ ]:
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
filas = []
for c in [2024, 2030, 2035]:
    filas.append(dict(cohorte=c, escenario="(i) serie actual", N=dd.N_t[c], J_M=dd.J[c] / 1e6,
                      **{f"TBP_C ({l})": V[("i", None, l, "C")][c] for l, _ in A_LIST},
                      **{f"TBP_A ({l})": V[("i", None, l, "A")][c] for l, _ in A_LIST}))
    for e in ESC10:
        for ver in ["ii", "iii"]:
            Jv = dd.J[c] if ver == "ii" else JRW[e][c]
            filas.append(dict(cohorte=c, escenario=f"({ver}) {e}", N=N_escenario(e)[c], J_M=Jv / 1e6,
                              **{f"TBP_C ({l})": V[(ver, e, l, "C")][c] for l, _ in A_LIST},
                              **{f"TBP_A ({l})": V[(ver, e, l, "A")][c] for l, _ in A_LIST}))
tabla10 = pd.DataFrame(filas)
ref = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
tabla10["TBP_C(A30) vs (i) %"] = [100 * (r["TBP_C (A30)"] / ref[r["cohorte"]] - 1) for _, r in tabla10.iterrows()]
display(tabla10.set_index(["cohorte", "escenario"]).round({"N": 0, "J_M": 3}).round(3))

# efecto aislado de reponderar J: (iii) vs (ii), mismo N
ef = pd.DataFrame({e: {c: 100 * (V[("iii", e, "A30", "C")][c] / V[("ii", e, "A30", "C")][c] - 1) for c in [2024, 2030, 2035]} for e in ESC10})
print("Efecto de reponderar J (iii vs ii, mismo N), TBP_C en %:"); display(ef.round(2).rename_axis("cohorte"))

# robustez del resultado (iii) a la convención de pesos, escenario persistencia, TBP_C (Ā = 30)
rob = {}
for nombre, kw in [("principal: DEIS 1950-2024", {}), ("solo 2023-2024 con DEIS", {"solo_2023_24": True}),
                   ("cohorte = prom. de y-a e y-a-1", {"mezcla": 0.5})]:
    Jv = J_rw_coh("persistencia", **kw)
    rob[nombre] = {c: tbp(N_escenario("persistencia"), Jv, A_LEGAL, "C")[c] for c in [2024, 2030, 2035]}
rob["(ii) J ONU"] = {c: V[("ii", "persistencia", "A30", "C")][c] for c in [2024, 2030, 2035]}
rob["(i) serie actual"] = {c: V[("i", None, "A30", "C")][c] for c in [2024, 2030, 2035]}
print("TBP_C (Ā=30), persistencia: sensibilidad de (iii) a la convención de pesos"); display(pd.DataFrame(rob).T.round(3).rename_axis("versión"))

### 10.7 Controles (asserts)

1. Con $w\equiv 1$ el método reproduce exactamente $J$ ONU de `d` en las 86 cohortes.
2. $J^{rw}>0$ y finito en todos los escenarios.
3. Para $t\le 2024$ todos los escenarios dan el mismo $J^{rw}$; para $t\ge 2025$ difieren.
4. Con nacimientos propios menores a los de la ONU, $J^{rw}\le J^{ONU\text{ después de }2024}$ para $t\ge 2025$.
5. (i)=(ii) hasta 2024; (iii)=(ii)·$J^{ONU}/J^{rw}$; linealidad en $\bar A$.
6. $w=1$ para $c<1950$ y 1971–74; $w\cdot N_{ONU}=N_t$ de `d` hasta 2024; un caso puntual coincide con la suma explícita.

In [ ]:
# 1) con w = 1 el método reproduce EXACTAMENTE J de la ONU en d, para todas las cohortes
J_w1 = J_reponderado(pesos_cohorte(None), ANIOS65); J_w1.index = dd.index
assert np.allclose(J_w1.values, dd.J.values, rtol=1e-12, atol=1e-6)
assert np.allclose(J_w1.values, (pop10.assign(e=pd.to_numeric(pop10.AgeGrpStart)).query("e >= 65")
                                 .groupby("Time").PopTotal.sum().loc[ANIOS65].values) * 1000)
print("OK 1) con w ≡ 1: J_rw = J de la ONU (d.J) en las 86 cohortes (y = suma PopTotal 65+ de la tabla de edad simple)")

# 2) J_rw > 0 en todos los escenarios y cohortes
for e, Jr in JRW.items():
    assert (Jr > 0).all() and np.isfinite(Jr.values).all(), e
print("OK 2) J_rw > 0 y finito en todos los escenarios y cohortes")

# 3) para t <= 2024 los escenarios no cambian J_rw (el stock 65+ de y=t+65 solo incluye cohortes c <= t <= 2024)
for e in ESC10[1:] + ["onu_medio"]:
    assert np.allclose(JRW[e].loc[:2024].values, JRW["persistencia"].loc[:2024].values, rtol=1e-12)
assert not np.allclose(JRW["recuperacion_lenta"].loc[2025:].values, JRW["persistencia"].loc[2025:].values)
print("OK 3) t <= 2024: J_rw idéntico en todos los escenarios; t >= 2025: los escenarios sí difieren")

# 4) 'onu_medio' (w = 1 desde 2025) y 'solo DEIS<=2024' coinciden con el método principal hasta t=2024 y, para t>=2025, J_rw(onu_medio) > J_rw(persistencia)
assert (JRW["onu_medio"].loc[2025:] >= JRW["persistencia"].loc[2025:] - 1e-6).all()
print("OK 4) con nacimientos propios menores a la ONU, J_rw(persistencia) <= J_rw(ONU después de 2024) para t >= 2025")

# 5) las versiones coinciden donde corresponde: (i)=(ii) para t<=2024 ; (ii)=(iii) NO coinciden (J cambia) ; (iii) usa el mismo N que (ii)
for e in ESC10:
    assert np.allclose(V[("i", None, "A30", "C")].loc[:2024], V[("ii", e, "A30", "C")].loc[:2024])
    assert np.allclose(V[("ii", e, "A30", "C")].loc[2025:] * dd.J.loc[2025:] / JRW[e].loc[2025:], V[("iii", e, "A30", "C")].loc[2025:])
assert np.allclose(V[("iii", "persistencia", "A14_2", "C")] * 30 / 14.2, V[("iii", "persistencia", "A30", "C")])   # linealidad en Ā
print("OK 5) (i)=(ii) hasta 2024; (iii)=(ii)·J_ONU/J_rw; TBP lineal en Ā")

# 6) peso w(c) = 1 para c < 1950, y la identidad sum(PopTotal·w) con w=DEIS/ONU en un caso puntual
w_p = pesos_cohorte("persistencia")
assert (w_p.loc[:1949] == 1).all() and np.isclose(w_p[2024], N_deis10[2024] / N_onu10[2024])
y0 = 2040
manual = sum(P.loc[y0, a] * w_p[y0 - a] for a in range(65, 101))
assert np.isclose(manual, J_reponderado(w_p, [y0])[y0])
assert (w_p.loc[1971:1974] == 1).all()                                    # hueco DEIS 1971-74: N_t de d usa ONU
assert np.allclose((w_p * N_onu10.reindex(w_p.index)).loc[1950:2024].values, dd.N_t.loc[1950:2024].values)   # w·N_ONU = N_t de d
print("OK 6) w(c)=1 para c<1950 y 1971-74 (hueco DEIS); w·N_ONU = N_t de d hasta 2024; cálculo puntual de J_rw(2040) coincide con la suma explícita")

### 10.8 Salida: `tbp_cohortes_J_reponderado.csv`

Formato largo, una fila por cohorte y escenario (`persistencia`, `recuperacion_lenta`, `caida_adicional` y `onu_medio` = DEIS hasta 2024 y ONU después, la referencia). Columnas: `J_onu`, `J_reponderado`, `N_propio`, y $TBP_C$ / $TBP_A$ para $\bar A=30$ y $14{,}2$ con $J$ reponderado (versión iii) y, con sufijo `_sinrepond`, con $J$ ONU (versión ii).

In [ ]:
partes = []
for e in ESC10 + ["onu_medio"]:
    t = pd.DataFrame({"cohorte": dd.index, "escenario": e, "anio_65": dd.anio_65.values,
                      "N_propio": (N_escenario(e) if e != "onu_medio" else dd.N_t).values,
                      "J_onu": dd.J.values, "J_reponderado": JRW[e].values})
    for ver, Jv, suf in [("iii", t.J_reponderado.values, ""), ("ii", t.J_onu.values, "_sinrepond")]:
        for A_lab, A in A_LIST:
            N = t.N_propio.values
            c = N * dd.S65.values * A * TAU / (Jv * RHO)
            t[f"TBP_C_{A_lab}{suf}"] = c
            t[f"TBP_A_{A_lab}{suf}"] = c / dd.E65.values
    partes.append(t)
out10 = pd.concat(partes, ignore_index=True)
assert out10.escenario.nunique() == 4 and len(out10) == 4 * len(dd)
DEST10 = next((p for p in ["../datos/procesados", "datos/procesados"] if os.path.isdir(p)), OUT)
os.makedirs(DEST10, exist_ok=True)
f10 = os.path.join(DEST10, "tbp_cohortes_J_reponderado.csv")
out10.round({"N_propio": 0, "J_onu": 0, "J_reponderado": 0}).round(6).to_csv(f10, index=False)
print("Escrito:", os.path.abspath(f10), out10.shape); print("Escrito:", f"{OUT}/seccion_10_J_reponderado.png y seccion_10_TBP_C_reponderado.png")
out10.head(3)

### 10.9 Lectura y límites

* **Qué cambia en la conclusión.** La curva post-2024 **sigue siendo más baja que la serie actual** con persistencia y con caída adicional, pero **menos** de lo que mostraba la versión inconsistente: reponderar $J$ sube $TBP_C$ de las cohortes recientes (cohorte 2035, $\bar A=30$, persistencia: 0,463 -> 0,513; serie actual 0,603). Con recuperación lenta la cohorte 2035 (0,623) queda ligeramente **por encima** de la serie actual (0,603): la ONU y ese escenario terminan en nacimientos parecidos y el $J$ menor empuja hacia arriba.
* **Tamaño del efecto.** Para $t\ge 2025$ $J$ cae hasta -9,9% (persistencia), -5,8% (recuperación) y -12,0% (caída) en la cohorte 2035. El rebote artificial de 2025 desaparece en los dos casos de natalidad baja.
* **Para $t\le 2024$** $J$ baja 2,4% en promedio (máx 4,8%, cohorte 1999), sobre todo por la diferencia de nivel DEIS-ONU en cohortes viejas, no por 2023-24; con la variante que ajusta solo 2023-24 la diferencia es 0,0% (máx 0,9%). El nivel de $TBP$ de las cohortes ≤2024 sube ~5% con el método principal y la forma de la curva no cambia.
* **Qué NO se puede decir:** que los escenarios sean pronósticos; que la reponderación sea exacta (supone mortalidad y migración proporcionales a la ONU, trata 100+ como una cohorte y usa $c=y-a$ pese a que la población es de mitad de año); que la corrección de nivel para cohortes viejas (DEIS vs ONU) sea inequívoca.
* **Pendiente:** validar los escenarios con el grupo y reemplazar el dato provisorio de 2025 del RENAPER por el definitivo.

## 11. 🆕 Sensibilidad (ρ, τ, N), hipótesis testeables y control

Esta sección **depende solo de variables ya en memoria** del notebook principal: `d` (cohortes 1950–2035), `s` (series por año), `sens` (sensibilidad de $\bar A$, sección 7), `TAU`, `RHO`, `A_OBS`, `A_LEGAL`, `DATA` y `OUT`. Escribe tablas y gráficos en `OUT/salidas/`.

**Qué es y qué no es.** El TBP es una **identidad determinística**: no hay variable resultado ni error aleatorio. Por eso **no corresponde un test de significancia**. La validación se hace por (i) **estática comparativa** (mover un parámetro y mirar cuánto cambia el índice), (ii) **escenarios** y (iii) **comparación con un indicador convencional** (razón de dependencia). No se afirma novedad.

Recordatorio algebraico: $TBP_C = \dfrac{N(1-\mu)\,\bar A\,\tau}{J\,\rho}$ (años), $TBP_A = TBP_C / E_r$, $TBP_A = D\times B$ con $D = N(1-\mu)/J$ y $B = \bar A\tau/(E_r\rho)$.

In [ ]:
import os
import numpy as np, pandas as pd, matplotlib.pyplot as plt

SAL = os.path.join(OUT, "salidas")          # CSV y PNG de esta seccion (NO datos/procesados)
os.makedirs(SAL, exist_ok=True)

# ---- Parametros con fuente: Anuario Estadistico ANSES 2008-2023 (datos_tbp/Anuario Estadístico_final.pdf) ----
#  p.91: tasa de sustitucion = haber medio SIPA (dic) / salario imponible medio (nov)
#  p.39: aporte personal 11%; contribucion patronal SIPA 10,77% (inc. a), 12,35% (inc. b), 16% (admin. publica)
ANSES = dict(rho_dic2009=0.371, rho_dic2023=0.403,
             rho_sin_moratoria_2009=0.425, rho_sin_moratoria_2023=0.563,
             aporte_personal=0.11, contrib_inc_a=0.1077, contrib_inc_b=0.1235, contrib_admin_publica=0.16)
TAU_A  = ANSES["aporte_personal"] + ANSES["contrib_inc_a"]          # 21,77 %
TAU_B  = ANSES["aporte_personal"] + ANSES["contrib_inc_b"]          # 23,35 %
TAU_AP = ANSES["aporte_personal"] + ANSES["contrib_admin_publica"]  # 27 %
RHO_09, RHO_23, RHO_SM = ANSES["rho_dic2009"], ANSES["rho_dic2023"], ANSES["rho_sin_moratoria_2023"]

# ---- Control 4c: los valores citados coinciden con los del notebook y con lo documentado ----
assert np.isclose(TAU, 0.2177) and np.isclose(TAU, TAU_A)
assert np.isclose(TAU_B, 0.2335) and np.isclose(TAU_AP, 0.27)
assert np.isclose(RHO, 0.403) and np.isclose(RHO, RHO_23)
assert np.isclose(RHO_09, 0.371) and np.isclose(RHO_SM, 0.563)
assert A_OBS == 14.2 and A_LEGAL == 30
print("OK 4c) tau=21,77% (23,35% / 27% alternativos), rho=40,3% (37,1% / 56,3% alternativos), A=14,2 y 30: coinciden con lo citado")

c = d.set_index("cohorte")
def tbp_C(N, S, J, A, tau, rho):            return N * S * A * tau / (J * rho)
def tbp_A(N, S, J, E, A, tau, rho):         return N * S * A * tau / (J * E * rho)

# consistencia con el notebook principal (CSV y seccion 7)
assert np.allclose(tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO), c.TBP_C_legal)
assert np.allclose(tbp_A(c.N_t, c.S65, c.J, c.E65, A_OBS, TAU, RHO), c.TBP_A_obs)
_chk = sens[(sens.A_barra == 30) & (sens.cohorte == 2024)].TBP_C.iloc[0]
assert np.isclose(_chk, c.TBP_C_legal[2024])
print(f"OK) funciones reproducen TBP_C/TBP_A del notebook principal; TBP_C(2024, A=30) = {c.TBP_C_legal[2024]:.3f} años")

### 11.1 Sensibilidad a $\rho$ y $\tau$ (y $\bar A$, $N$)

**Rangos justificados** (todos con fuente, salvo lo rotulado como ilustración):

* $\rho$: 0,371 (dic-2009), 0,403 (dic-2023, **base**) y 0,563 (haber contributivo **sin moratoria**, 2023) — Anuario ANSES p. 91.
* $\tau$: 0,2177 (11% + 10,77%, **base**), 0,2335 (11% + 12,35%, inciso b) y 0,27 (11% + 16%, administración pública) — Anuario ANSES p. 39.
* $\bar A$: 15 y 30 (30 es un requisito legal de referencia, **no un máximo**; 14,2 es un valor observado sesgado).

**Qué esperar:** $TBP_C$ es **lineal** en $\bar A$, $\tau$ y $1/\rho$ ⇒ estos parámetros mueven el **nivel** de toda la curva por un factor constante, no su forma. Lo que *sí* cambiaría la forma es que un parámetro **dependa de la cohorte** (ejemplo ilustrativo más abajo, **no es un dato**).

In [ ]:
RHOS = [RHO_09, RHO, RHO_SM]; TAUS = [TAU, TAU_B, TAU_AP]; AS = [15.0, 30.0]; COH = [2000, 2010, 2024]
filas = []
for t in COH:
    for A in AS:
        for rho in RHOS:
            for tau in TAUS:
                filas.append(dict(cohorte=t, A_barra=A, rho=rho, tau=tau,
                                  TBP_C=tbp_C(c.N_t[t], c.S65[t], c.J[t], A, tau, rho)))
grid = pd.DataFrame(filas)

# ---- Control 4a: linealidad en rho (TBP*rho constante) y en tau (TBP/tau constante) y en A ----
for (t, A), g in grid.groupby(["cohorte", "A_barra"]):
    assert np.allclose(g.TBP_C * g.rho / g.tau, g.TBP_C.iloc[0] * g.rho.iloc[0] / g.tau.iloc[0])   # TBP*rho/tau = const
for rho in RHOS:
    for tau in TAUS:
        g = grid[(grid.rho == rho) & (grid.tau == tau)].pivot(index="cohorte", columns="A_barra", values="TBP_C")
        assert np.allclose(g[30.0], 2 * g[15.0])                       # lineal en A
base_ = lambda t: tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO)
for t in COH:
    for k in (0.5, 2.0):                                                # escalar tau o 1/rho por k escala TBP por k
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, k * TAU, RHO), k * base_(t))
        assert np.isclose(tbp_C(c.N_t[t], c.S65[t], c.J[t], A_LEGAL, TAU, RHO / k), k * base_(t))
print("OK 4a) linealidad: TBP_C*rho/tau constante, TBP_C(k*tau)=k*TBP_C, TBP_C(rho/k)=k*TBP_C, TBP_C(2A)=2*TBP_C")

tabla = grid.pivot_table(index=["A_barra", "cohorte", "rho"], columns="tau", values="TBP_C")
tabla.columns = [f"tau={x:.4f}" for x in tabla.columns]
print("TBP_C (años) por cohorte x Ā x ρ x τ  (tau=0,2177 base; 0,2335 inc. b; 0,27 admin. pública):")
display(tabla.round(3))

rng = grid.groupby("cohorte").TBP_C.agg(["min", "max"]); rng["max/min"] = rng["max"] / rng["min"]
print("\nRango de TBP_C en toda la grilla (3 ρ × 3 τ × 2 Ā):"); display(rng.round(3))
print("Solo ρ (τ y Ā base, Ā=30): TBP_C(ρ=0,563)/TBP_C(ρ=0,371) =", round(RHO_09 / RHO_SM, 3), "-> factor constante en todas las cohortes")
tabla.reset_index().to_csv(os.path.join(SAL, "s11_grilla_rho_tau_A.csv"), index=False)

In [ ]:
# ---- Curvas por cohorte para 3 valores de rho (Ā=30, tau base) + ejemplo ILUSTRATIVO no lineal ----
coh = c.index.values
fig, ax = plt.subplots(1, 3, figsize=(19, 5.2))
for rho, col, lab in zip(RHOS, ["tab:green", "tab:blue", "tab:red"],
                         ["ρ=0,371 (dic-2009)", "ρ=0,403 (dic-2023, base)", "ρ=0,563 (sin moratoria 2023)"]):
    ax[0].plot(coh, tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho), color=col, label=lab)
ax[0].set_title("TBP_C por cohorte según ρ (Ā=30, τ=0,2177)\nρ constante ⇒ cambia el NIVEL, no la forma", fontsize=10)

# ILUSTRATIVO: rho cae para cohortes jovenes por menor cobertura. NO ES UN DATO.
def rho_ilus(t):
    return RHO * (1 - 0.40 * np.clip((np.asarray(t, dtype=float) - 1975) / (2000 - 1975), 0, 1))
tb_const = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, RHO)
tb_ilus  = tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, rho_ilus(coh))
ax[1].plot(coh, tb_const, color="tab:blue", label="ρ constante = 0,403")
ax[1].plot(coh, tb_ilus, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[1].set_title("EJEMPLO ILUSTRATIVO — NO ES DATO: ρ cae 40% entre\ncohortes 1975 y 2000 (supuesto inventado, menor cobertura)", fontsize=10, color="firebrick")
idx_c = 100 * tb_const / tb_const[1950]; idx_i = 100 * tb_ilus / tb_ilus[1950]
ax[2].plot(coh, idx_c, color="tab:blue", label="ρ constante (índice 1950=100)")
ax[2].plot(coh, idx_i, color="tab:orange", ls="--", label="ρ(cohorte) ILUSTRATIVO")
ax[2].set_title("Índice 1950=100: con ρ(cohorte) la FORMA sí cambia\n(ilustrativo, no es dato)", fontsize=10, color="firebrick")
for a in ax:
    a.axvline(2024.5, color="gray", ls=":"); a.grid(alpha=.3); a.legend(fontsize=8); a.set_xlabel("año de nacimiento de la cohorte")
ax[0].set_ylabel("TBP_C (años)")
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_curvas_rho.png"), dpi=130); plt.show()

# Control: con rho constante el indice normalizado es identico entre los 3 rho; con rho(cohorte) NO lo es
idx_rho = [100 * tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r) / tbp_C(c.N_t, c.S65, c.J, A_LEGAL, TAU, r)[1950] for r in RHOS]
assert all(np.allclose(idx_rho[0], x) for x in idx_rho[1:])
assert np.ptp(idx_i - idx_c) > 5          # la forma cambia de manera apreciable
print(f"OK) índice 2024 (1950=100): rho constante = {idx_c[2024]:.1f} (igual para cualquier rho) | rho(cohorte) ilustrativo = {idx_i[2024]:.1f}")
print("Lectura: TBP menor ρ ⇒ MAYOR TBP (la prestación es más barata respecto del aporte); TBP alto no significa 'mejor' para el jubilado.")

In [ ]:
# ---- Tornado para la cohorte 2024: variacion % de TBP_C al mover cada parametro dentro de su rango ----
t0 = 2024
N0, S0, J0 = c.N_t[t0], c.S65[t0], c.J[t0]
base = tbp_C(N0, S0, J0, A_LEGAL, TAU, RHO)          # base: A=30 (referencia legal), rho=0,403, tau=0,2177
rangos = [   # (nombre, valor bajo, valor alto, funcion TBP(valor))
    ("Ā  [15 – 30]",                         15.0, 30.0,       lambda v: tbp_C(N0, S0, J0, v, TAU, RHO)),
    ("ρ  [0,371 – 0,563]",                   RHO_SM, RHO_09,   lambda v: tbp_C(N0, S0, J0, A_LEGAL, TAU, v)),   # rho alto => TBP bajo
    ("τ  [0,2177 – 0,27]",                   TAU, TAU_AP,      lambda v: tbp_C(N0, S0, J0, A_LEGAL, v, RHO)),
    ("J  ±10% (ILUSTRATIVO, proyección ONU)", 1.10 * J0, 0.90 * J0, lambda v: tbp_C(N0, S0, v, A_LEGAL, TAU, RHO)),
    ("N  ±10% (ILUSTRATIVO)",                0.90 * N0, 1.10 * N0, lambda v: tbp_C(v, S0, J0, A_LEGAL, TAU, RHO)),
]
torn = pd.DataFrame([dict(parametro=n, valor_en_extremo_bajo_TBP=lo, valor_en_extremo_alto_TBP=hi,
                          var_pct_bajo=100 * (f(lo) / base - 1), var_pct_alto=100 * (f(hi) / base - 1))
                     for n, lo, hi, f in rangos])
torn["amplitud_pp"] = torn.var_pct_alto - torn.var_pct_bajo
torn = torn.sort_values("amplitud_pp")
display(torn.round(2))

fig, ax = plt.subplots(figsize=(9, 4.2))
y = np.arange(len(torn))
ax.barh(y, torn.var_pct_bajo, color="tab:red", alpha=.8, label="extremo que baja TBP")
ax.barh(y, torn.var_pct_alto, color="tab:blue", alpha=.8, label="extremo que sube TBP")
ax.set_yticks(y); ax.set_yticklabels(torn.parametro, fontsize=8.5); ax.axvline(0, color="k", lw=.8)
ax.set_xlabel(f"variación % de TBP_C (cohorte 2024)\nbase = {base:.2f} años (Ā=30, ρ=0,403, τ=0,2177)", fontsize=9)
ax.set_title("Tornado: cohorte 2024 (N de DEIS; J y S65 proyección ONU)", fontsize=10); ax.grid(alpha=.3, axis="x"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_tornado_cohorte2024.png"), dpi=130); plt.show()
# controles aritmeticos del tornado (lineales / hiperbolica)
assert np.isclose(torn.set_index("parametro").loc["Ā  [15 – 30]", "var_pct_bajo"], -50.0)
assert np.isclose(torn.set_index("parametro").loc["N  ±10% (ILUSTRATIVO)", "var_pct_bajo"], -10.0)
assert np.isclose(torn.set_index("parametro").loc["ρ  [0,371 – 0,563]", "var_pct_bajo"], 100 * (RHO / RHO_SM - 1))
torn.to_csv(os.path.join(SAL, "s11_tornado_cohorte2024.csv"), index=False)
print("Nota: la base usa Ā=30 (referencia legal, no un máximo); con otra base de Ā los porcentajes de Ā cambian, los de ρ, τ y N no.")

### 11.2 Hipótesis testeables y estática comparativa

Cada hipótesis se formula con **signo esperado**, **criterio de rechazo** y **la parte del modelo** que la pone a prueba. **Honestidad metodológica:** como el TBP es una identidad, las "pruebas" que siguen son **comparaciones de estática comparativa** (¿el signo y la magnitud calculados con las cohortes reales coinciden con lo esperado?), **no pruebas estadísticas**: no hay p-valores ni intervalos de confianza, y las cohortes sucesivas no son observaciones independientes.

| | Hipótesis | Signo esperado | Parte del modelo | Criterio de rechazo |
|---|---|---|---|---|
| H1 | Una caída sostenida de $N$ respecto de las cohortes que integran $J$ reduce el TBP | − (vía $D$) | numerador $N$ y denominador $J$ de $D$ | La contribución de $\Delta\ln N$ a $\Delta\ln TBP$ tiene signo contrario, o $N$ cae y TBP no cae con el resto fijo |
| H2 | Mayor longevidad ($E_r$, $J$) reduce el TBP | − (doble canal en $TBP_A$; un canal en $TBP_C$) | $E_r$ y $J$ | La elasticidad de $TBP_A$ a $E_r$ no es −1, o la contribución de $-\Delta\ln J$ no es negativa cuando $J$ crece |
| H3 | Mayor densidad de aportes $\bar A$ aumenta el TBP (**nivel, no forma**) | + | $\bar A$ | Elasticidad ≠ +1, o el índice normalizado cambia con $\bar A$ |
| H4 | Mayor formalidad (más $\bar A$ y/o $\tau$ efectiva) aumenta el TBP | + | $\bar A$, $\tau$ | Igual que H3/H5; **la formalidad por cohorte requiere EPH (sección 8, sin datos reales todavía)** |
| H5 | Mayor tasa de aporte $\tau$ aumenta el TBP | + | $\tau$ | Elasticidad ≠ +1 |
| H6 | Mayor tasa de reemplazo $\rho$ reduce el TBP | − | $\rho$ | Elasticidad ≠ −1 |

Nota: $S_{65}=1-\mu$ (supervivencia hasta los 65) **aumenta** el TBP: más longevidad *antes* de 65 y más longevidad *después* de 65 empujan en sentidos opuestos. Por eso H2 se refiere a la longevidad **posterior** a los 65 ($E_r$) y al stock de mayores $J$.

In [ ]:
# ---- Elasticidades discretas (arco, +10%) en la cohorte 2024: eps = ln(TBP1/TBP0) / ln(x1/x0) ----
t0 = 2024
P0 = dict(N=c.N_t[t0], S=c.S65[t0], J=c.J[t0], E=c.E65[t0], A=A_LEGAL, tau=TAU, rho=RHO)
def f_C(p): return tbp_C(p["N"], p["S"], p["J"], p["A"], p["tau"], p["rho"])
def f_A(p): return tbp_A(p["N"], p["S"], p["J"], p["E"], p["A"], p["tau"], p["rho"])
esp = {"N": (1, 1), "S": (1, 1), "J": (-1, -1), "E": (0, -1), "A": (1, 1), "tau": (1, 1), "rho": (-1, -1)}
et = []
for k in P0:
    p1 = dict(P0); p1[k] = 1.10 * P0[k]
    eC = np.log(f_C(p1) / f_C(P0)) / np.log(1.10); eA = np.log(f_A(p1) / f_A(P0)) / np.log(1.10)
    assert np.isclose(eC, esp[k][0]) and np.isclose(eA, esp[k][1])
    et.append(dict(parametro=k, elasticidad_TBP_C=eC, elasticidad_TBP_A=eA, hipotesis={"N":"H1","S":"-","J":"H1/H2","E":"H2","A":"H3/H4","tau":"H4/H5","rho":"H6"}[k]))
elas = pd.DataFrame(et)
display(elas.round(3))
print("OK) signos y magnitudes de H1–H6 coinciden con lo esperado (elasticidades ±1; E_r: 0 en TBP_C, −1 en TBP_A).")
print("Ojo: estas elasticidades son propiedades del diseño del índice (cada factor entra con exponente ±1), no un resultado empírico.")

In [ ]:
# ---- Descomposición logarítmica EXACTA de la caída del TBP entre cohortes ----
#  ln TBP_C = ln N + ln S65 + ln Ā + ln τ − ln J − ln ρ  =>  con Ā, τ, ρ constantes:
#  Δln TBP_C = Δln N + Δln S65 − Δln J ;   Δln TBP_A = Δln TBP_C − Δln E_r
def descomponer(t0, t1, tipo="C"):
    comp = {"N (natalidad)": np.log(c.N_t[t1] / c.N_t[t0]),
            "S65 (supervivencia a 65)": np.log(c.S65[t1] / c.S65[t0]),
            "J (stock 65+; envejecimiento)": -np.log(c.J[t1] / c.J[t0])}
    if tipo == "A": comp["E_r (longevidad a los 65)"] = -np.log(c.E65[t1] / c.E65[t0])
    col = "TBP_C_legal" if tipo == "C" else "TBP_A_legal"
    total = np.log(c[col][t1] / c[col][t0])           # calculado desde el indice, independiente de los componentes
    assert np.isclose(sum(comp.values()), total, atol=1e-12)                      # control 4b
    sh = {k: 100 * v / total for k, v in comp.items()}
    assert np.isclose(sum(sh.values()), 100.0)
    return comp, total, sh

res = []
for (t0_, t1_) in [(1990, 2024), (2000, 2024), (2010, 2024)]:
    for tipo in "CA":
        comp, total, sh = descomponer(t0_, t1_, tipo)
        row = dict(desde=t0_, hasta=t1_, indice="TBP_" + tipo, dlnTBP=total, variacion_pct=100 * (np.exp(total) - 1))
        for k in comp: row["dln " + k] = comp[k]; row["% " + k] = sh[k]
        row["suma_%"] = sum(sh.values()); res.append(row)
desc = pd.DataFrame(res)
print("OK 4b) la suma de las contribuciones = ΔlnTBP (100%) en todas las comparaciones")
display(desc[["desde", "hasta", "indice", "variacion_pct", "dlnTBP"] + [x for x in desc.columns if x.startswith("% ")] + ["suma_%"]].round(2))

comp, total, sh = descomponer(2000, 2024, "C")
print(f"\nTBP_C cohorte 2000 -> 2024: {c.TBP_C_legal[2000]:.3f} -> {c.TBP_C_legal[2024]:.3f} años ({100*(np.exp(total)-1):.1f}%); Δln = {total:.4f}")
for k in comp: print(f"   {k:34s} Δln = {comp[k]:+.4f}  -> {sh[k]:6.1f}% de la caída")

# Detalle de H1: N de la cohorte vs. promedio de N de las cohortes que integran J (nacidas t-35..t, 65 a 100 años en t+65)
Nref = pd.Series({t: c.N_t.loc[t - 35:t].mean() for t in range(1985, 2036)})
for (t0_, t1_) in [(2000, 2024)]:
    rel = np.log((c.N_t[t1_] / Nref[t1_]) / (c.N_t[t0_] / Nref[t0_])); ref = np.log(Nref[t1_] / Nref[t0_])
    assert np.isclose(rel + ref, np.log(c.N_t[t1_] / c.N_t[t0_]))
    print(f"\nDetalle H1 ({t0_}->{t1_}): Δln N = {comp['N (natalidad)']:+.4f} = {rel:+.4f} (N relativo al promedio de las 36 cohortes que integran J) + {ref:+.4f} (nivel de ese promedio)")
print("Lectura: J agrega natalidad y longevidad de cohortes anteriores; por eso la contribución de J no es 'longevidad pura'.")
desc.to_csv(os.path.join(SAL, "s11_descomposicion_log.csv"), index=False)
elas.to_csv(os.path.join(SAL, "s11_elasticidades_cohorte2024.csv"), index=False)

fig, ax = plt.subplots(figsize=(8, 4))
comps = {"C": descomponer(2000, 2024, "C")[0], "A": descomponer(2000, 2024, "A")[0]}
labels = list(comps["A"].keys()); xx = np.arange(len(labels)); w = .38
ax.bar(xx - w/2, [comps["C"].get(k, 0) for k in labels], w, label="TBP_C (recomendada)", color="tab:blue")
ax.bar(xx + w/2, [comps["A"][k] for k in labels], w, label="TBP_A (con E_r)", color="tab:orange")
ax.axhline(0, color="k", lw=.8); ax.set_xticks(xx); ax.set_xticklabels(labels, fontsize=8, rotation=10)
ax.set_ylabel("contribución a Δln TBP (2000 → 2024)"); ax.set_title("Descomposición logarítmica exacta de la caída 2000→2024", fontsize=10)
ax.grid(alpha=.3, axis="y"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(SAL, "s11_descomposicion_2000_2024.png"), dpi=130); plt.show()

In [ ]:
# ---- Tabla de hipotesis con la evidencia numerica de estatica comparativa ----
cC, tC, shC = descomponer(2000, 2024, "C"); cA, tA, shA = descomponer(2000, 2024, "A")
e = elas.set_index("parametro")
cf_N = tbp_C(c.N_t[2000], c.S65[2024], c.J[2024], A_LEGAL, TAU, RHO) / c.TBP_C_legal[2024] - 1
hip = pd.DataFrame([
 dict(id="H1", hipotesis="Caída sostenida de N respecto de las cohortes que integran J reduce TBP (vía D)", signo="−",
      parte_del_modelo="N y J en D", criterio_rechazo="Contribución de Δln N con signo opuesto, o N cae y TBP_C no cae (resto fijo)",
      evidencia=f"2000→2024: N {c.N_t[2000]:,.0f}→{c.N_t[2024]:,.0f}; contribución de N = {shC['N (natalidad)']:.1f}% de la caída de TBP_C; con N de 2000 y el resto de 2024, TBP_C(2024) sería {100*cf_N:+.0f}% mayor",
      estado="no rechazada (comparación de estática comparativa)"),
 dict(id="H2", hipotesis="Mayor longevidad (E_r y J) reduce TBP (doble canal en TBP_A, uno en TBP_C)", signo="−",
      parte_del_modelo="E_r y J", criterio_rechazo="Elasticidad de TBP_A a E_r ≠ −1 o contribución de J no negativa cuando J crece",
      evidencia=f"elasticidad TBP_A a E_r = {e.loc['E','elasticidad_TBP_A']:.0f}, a J = {e.loc['J','elasticidad_TBP_A']:.0f}; 2000→2024: contrib. de J = {shC['J (stock 65+; envejecimiento)']:.1f}% (TBP_C), de E_r = {shA['E_r (longevidad a los 65)']:.1f}% (TBP_A); S65 empuja en sentido contrario ({shC['S65 (supervivencia a 65)']:.1f}%)",
      estado="no rechazada (E_r casi no cambia; el canal J es el relevante)"),
 dict(id="H3", hipotesis="Mayor densidad de aportes Ā aumenta TBP (nivel, no forma)", signo="+",
      parte_del_modelo="Ā", criterio_rechazo="Elasticidad ≠ +1 o el índice normalizado depende de Ā",
      evidencia=f"elasticidad = {e.loc['A','elasticidad_TBP_C']:.0f}; Ā 15→30 duplica TBP_C; índice 1950=100 idéntico (sección 7)", estado="no rechazada (propiedad del diseño)"),
 dict(id="H4", hipotesis="Mayor formalidad = mayor Ā y/o τ efectiva ⇒ TBP mayor", signo="+",
      parte_del_modelo="Ā y τ", criterio_rechazo="Como H3/H5; requiere Ā por cohorte (EPH, sección 8)",
      evidencia=f"efecto de signo conocido (+); ejemplo: Ā 15→22,5 (+50%) y τ 0,2177→0,27 (+24%) ⇒ TBP_C ×{1.5*TAU_AP/TAU:.2f}", estado="signo verificado; magnitud NO evaluable sin datos EPH"),
 dict(id="H5", hipotesis="Mayor τ aumenta TBP", signo="+", parte_del_modelo="τ", criterio_rechazo="Elasticidad ≠ +1",
      evidencia=f"elasticidad = {e.loc['tau','elasticidad_TBP_C']:.0f}; τ 0,2177→0,27: {100*(TAU_AP/TAU-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
 dict(id="H6", hipotesis="Mayor ρ reduce TBP", signo="−", parte_del_modelo="ρ", criterio_rechazo="Elasticidad ≠ −1",
      evidencia=f"elasticidad = {e.loc['rho','elasticidad_TBP_C']:.0f}; ρ 0,403→0,563: {100*(RHO/RHO_SM-1):+.1f}%; 0,403→0,371: {100*(RHO/RHO_09-1):+.1f}%", estado="no rechazada (propiedad del diseño)"),
])
pd.set_option("display.max_colwidth", 200)
display(hip[["id", "signo", "parte_del_modelo", "evidencia", "estado"]])
hip.to_csv(os.path.join(SAL, "s11_hipotesis.csv"), index=False)
print("Aclaración: 'no rechazada' = el signo y la magnitud calculados coinciden con lo esperado; NO es un resultado estadístico.")

### 11.3 Control con un indicador convencional: razón de dependencia (ONU, 65+ / 15–64)

La razón de dependencia de vejez es el indicador habitual del envejecimiento. Se compara, **por cohorte $t$**, el $TBP_C$ con la razón de dependencia en el año $t+65$ (cuando la cohorte tiene 65, el momento que pondera el TBP) y en $t+40$ (mitad de la vida activa). Esperado: **signo negativo** (más dependencia ⇒ menor TBP).

**⚠ Circularidad mecánica.** $J$ (población 65+ en $t+65$) es el numerador de la razón de dependencia **en $t+65$** y el denominador de $TBP_C$: un signo negativo en $t+65$ está **garantizado en parte por construcción**. La comparación en $t+40$ evita esa coincidencia exacta de año, pero ambas series tienen **tendencia** (envejecimiento), de modo que una correlación de rangos entre series con tendencia es casi inevitable. Se agregan dos diagnósticos: (i) la correlación de la razón con $1/J$ solo (benchmark mecánico) y (ii) las **primeras diferencias** (cambios entre cohortes consecutivas). **No se reportan p-valores** (cohortes consecutivas no son independientes).

In [ ]:
dep = s["un_dependencia_65mas_15_64"]
# la columna de la ONU = pob 65+ / pob 15-64 (control de consistencia con las series de poblacion)
chk = (s.un_pob_65mas_miles / s.un_pob_15_64_miles).dropna()
assert np.allclose(dep.loc[chk.index], chk, atol=1e-3)

x = pd.DataFrame({"cohorte": c.index, "TBP_C": c.TBP_C_legal.values, "TBP_B": c.TBP_B_legal.values,
                  "inv_J": 1 / c.J.values, "NS65": (c.N_t * c.S65).values})
x["dep_t65"] = dep.reindex(c.anio_65.values).values
x["dep_t40"] = dep.reindex((c.index.values + 40)).values
x = x.dropna()

def sp(a, b, df=x): return df[a].corr(df[b], method="spearman")
def sp_dif(a, b, df=x): return df[a].diff().corr(df[b].diff(), method="spearman")
filas = []
for etiqueta, df_ in [("todas las cohortes 1950–2035", x), ("solo cohortes con N observado (≤2024)", x[x.cohorte <= 2024])]:
    for h in ["dep_t65", "dep_t40"]:
        filas.append(dict(muestra=etiqueta, razon_dependencia_en=h.replace("dep_t", "t+"), n=len(df_),
                          spearman_TBP_C=sp("TBP_C", h, df_), spearman_inv_J_benchmark_mecanico=sp("inv_J", h, df_),
                          spearman_NxS65_sin_J=sp("NS65", h, df_), spearman_TBP_B_sin_N_ni_J=sp("TBP_B", h, df_),
                          spearman_primeras_dif_TBP_C=sp_dif("TBP_C", h, df_)))
corr = pd.DataFrame(filas)
display(corr.round(3))
corr.to_csv(os.path.join(SAL, "s11_correlacion_dependencia.csv"), index=False)

r65 = sp("TBP_C", "dep_t65"); r40 = sp("TBP_C", "dep_t40")
print(f"Spearman(TBP_C, razón de dependencia en t+65) = {r65:+.3f}   <- inflado por circularidad (J en ambos)")
print(f"Spearman(TBP_C, razón de dependencia en t+40) = {r40:+.3f}")
print(f"Benchmark mecánico: Spearman(1/J, razón en t+65) = {sp('inv_J','dep_t65'):+.3f}")
assert r65 < 0 and r40 < 0
print("OK) signo negativo en ambos horizontes, como se esperaba.")
print("Lectura prudente: el signo es el esperado, pero no es una validación independiente (circularidad y tendencia común). "
      "La información adicional del TBP respecto de la razón de dependencia está en N, S65, E_r, τ, ρ y Ā, que la razón no incluye.")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))
for a, h, ttl in [(ax[0], "dep_t65", "razón de dependencia en t+65 (circular)"), (ax[1], "dep_t40", "razón de dependencia en t+40")]:
    sc = a.scatter(x[h], x.TBP_C, c=x.cohorte, cmap="viridis", s=18)
    a.set_xlabel(ttl); a.set_ylabel("TBP_C (Ā=30, años)"); a.grid(alpha=.3)
    a.set_title(f"Spearman = {sp('TBP_C', h):+.2f}", fontsize=10)
fig.colorbar(sc, ax=ax, label="cohorte")
plt.savefig(os.path.join(SAL, "s11_tbp_vs_dependencia.png"), dpi=130, bbox_inches="tight"); plt.show()

### 11.4 Qué se puede y qué no se puede concluir

* **Sí:** el signo y el orden de magnitud de cada canal (H1–H6) con cohortes reales; que la caída del $TBP_C$ 2000→2024 se descompone **exactamente** en natalidad, supervivencia a 65 y stock de mayores; que $\rho$, $\tau$ y $\bar A$ cambian el **nivel** y no la forma.
* **No:** probar causalidad ni significancia; afirmar que $\rho$ o $\tau$ sean constantes entre cohortes (hoy se usa un valor único de 2023); evaluar H4 sin $\bar A$ por cohorte (EPH, sección 8, sin datos reales aún).
* **Cohorte 2024 y siguientes:** $J$, $S_{65}$ y $E_r$ son **proyecciones ONU a 2089+**: la incertidumbre demográfica es grande y no está en el tornado salvo la ilustración de $J$ ±10%.
* Los ejemplos rotulados "ILUSTRATIVO" (ρ dependiente de la cohorte, $N$ y $J$ ±10%) **no son datos**.

In [ ]:
# ---- Controles finales (4a, 4b, 4c ya ejecutados arriba) y listado de salidas ----
assert os.path.isdir(SAL) and "procesados" not in os.path.abspath(SAL)
for f in sorted(os.listdir(SAL)):
    if f.startswith("s11_"): print("Escrito:", os.path.join(SAL, f))
print("\nControles de la sección 11: 4a linealidad OK | 4b descomposición suma ΔlnTBP OK | 4c parámetros citados OK")

## 12. 🆕 Calibración de $\bar A$ con microdatos de la EPH: descarga, filtro de aportantes y edad de ingreso

Esta sección convierte el método de la sección 8 en una **rutina lista para correr con datos reales**. Pasos:

1. **Descargar / cargar** las bases usuarias individuales trimestrales de la EPH (INDEC). La descarga está **apagada por defecto** (`ACEPTO_DESCARGA = False`): primero muestra qué archivos son y cuánto pesan.
2. **Filtrar** la población 18–64 y marcar como *registrados* a los ocupados con descuento jubilatorio (`PP07H`).
3. **Pseudo-panel** por cohorte (`ANO4 − CH06`) y edad, y modelo logit aditivo cohorte + edad.
4. **Edad efectiva de ingreso al empleo registrado** por cohorte y **$\bar A$** por cohorte, con tres reglas para cuentapropistas.
5. **Conexión con el TBP:** $TBP_C$ con $\bar A(t)$ por cohorte en lugar de un valor único (la "Opción 3" de la propuesta del grupo, apenas haya datos reales).

> ⚠️ **Estado:** sin microdatos cargados esta sección **no calcula ningún $\bar A$ para Argentina**; sólo corre la prueba con **datos sintéticos** (12.6). Disponibilidad verificada de las bases: 2017-T2 a 2025-T4 (35 archivos, ~107 MB). La EPH es urbana y transversal, no observa la edad del primer empleo registrado, y el descuento jubilatorio mide la semana de referencia: ver 12.7.

### 12.1 Módulo (copia de `src/tbp_eph.py`)

Todo el código está en una sola celda para que el Colab sea autosuficiente. El mismo archivo vive en el repositorio (`src/tbp_eph.py`) y se puede usar desde la terminal: `python src/tbp_eph.py --plan`, `--descargar`, `--autotest`.

In [ ]:
# ====== MÓDULO tbp_eph (misma versión que src/tbp_eph.py) ======
# -*- coding: utf-8 -*-
"""
tbp_eph.py — Calibración de Ā (años de aporte esperados por persona de la cohorte) con la EPH (INDEC).

Rutina modular para el proyecto TBP — Índice de Cuna Vacía.

Flujo:
    1. plan_descarga / descargar_eph   -> baja las bases usuarias individuales trimestrales (con confirmación explícita)
    2. find_eph_files / read_all       -> lee las bases (txt o zip de INDEC)
    3. add_flags                       -> filtra ocupados y marca "registrado" (descuento jubilatorio PP07H)
    4. build_cells + ajuste logit      -> pseudo-panel cohorte x edad (densidad de aportes)
    5. edad_ingreso_*                  -> edad efectiva de ingreso al empleo registrado, por cohorte
    6. estimar_abar                    -> Ā por cohorte (3 reglas para independientes)
    7. tbp_con_abar                    -> TBP_C y TBP_A usando Ā(t) en lugar de un valor único

AVISOS METODOLÓGICOS (leer):
  * La EPH es una encuesta transversal y urbana: NO observa la edad del primer empleo registrado ni la historia de aportes
    de cada persona. Se trabaja con un pseudo-panel (cohorte = ANO4 − CH06) y se asume que el perfil por edad es estable
    entre cohortes (modelo logit aditivo cohorte + edad). Edad, período y cohorte no se pueden separar del todo.
  * PP07H ("¿le descuentan jubilación?") se pregunta a asalariados y mide la semana de referencia, no el año.
    Una persona con descuento no necesariamente suma "años de aporte" de ley (moratorias, regímenes provinciales, etc.).
  * La base individual no trae una variable inequívoca de aporte de cuentapropistas: por eso hay tres reglas
    ("solo_asalariados" = cota inferior, "pp07i", "escenario") y se reportan las tres.
  * Los nombres de variables y la lectura están basados en el diseño de registro de INDEC; verificar con la primera
    descarga real (ver `resumen_base`).
  * No hay valor de Ā "verdadero" para Argentina hasta correr esto con datos reales. Los números del autotest son sintéticos.

Uso mínimo:
    import tbp_eph as e
    e.descargar_eph("datos/eph")                                   # sólo muestra el plan (no descarga)
    e.descargar_eph("datos/eph", confirmado=True)                  # descarga de verdad
    df = e.read_all(e.find_eph_files(["datos/eph"]))
    tabla = e.estimar_abar(df, s65)                                # s65: Series cohorte -> supervivencia a 65
"""
import os, re, io, glob, time, zipfile, warnings, tempfile, urllib.request
import numpy as np
import pandas as pd

# ------------------------------------------------------------------ constantes
EDAD_MIN, EDAD_MAX = 18, 64
EDADES = np.arange(EDAD_MIN, EDAD_MAX + 1)               # 47 edades simples
BASE_URL = "https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph"
# Disponibilidad verificada (2 oct 2026, por cabecera HTTP y firma ZIP): 2017-T2 a 2025-T4.
# 2016 y 2017-T1 NO están con este patrón de nombre en el sitio actual de INDEC.
DISPONIBLE_DESDE, DISPONIBLE_HASTA = (2017, 2), (2025, 4)
USER_AGENT = "Mozilla/5.0 (compatible; TBP-proyecto-universitario)"
MIN_N_CELDA = 20          # mínimo de observaciones (sin ponderar) por celda cohorte x edad
MIN_EDADES_COHORTE = 3    # mínimo de edades observadas para estimar el efecto propio de una cohorte
GOMPERTZ_B = 0.085        # pendiente de la mortalidad adulta (supuesto)
IND_RULE_DEFAULT = "solo_asalariados"
IND_SHARE_DEFAULT = None
NEEDED = {"ANO4", "TRIMESTRE", "CH04", "CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"}
OBLIGATORIAS = {"ANO4", "CH06", "ESTADO", "PONDERA"}


# ================================================================== 1) DESCARGA
def urls_eph(desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA):
    """Lista de bases trimestrales (año, trimestre, nombre, url) en el rango pedido."""
    out = []
    for y in range(desde[0], hasta[0] + 1):
        for t in (1, 2, 3, 4):
            if (y, t) < tuple(desde) or (y, t) > tuple(hasta):
                continue
            nombre = f"EPH_usu_{t}_Trim_{y}_txt.zip"
            out.append({"anio": y, "trim": t, "nombre": nombre, "url": f"{BASE_URL}/{nombre}"})
    return out


def _ctx_ssl():
    """Contexto SSL con verificación ACTIVA. Usa certifi si está instalado (Python de macOS suele no traer certificados raíz)."""
    import ssl
    try:
        import certifi
        return ssl.create_default_context(cafile=certifi.where())
    except Exception:
        return ssl.create_default_context()


def _abrir(url, metodo="GET", rango=None, timeout=60):
    req = urllib.request.Request(url, method=metodo, headers={"User-Agent": USER_AGENT})
    if rango:
        req.add_header("Range", rango)
    return urllib.request.urlopen(req, timeout=timeout, context=_ctx_ssl())


def _curl(url, destino=None, solo_cabecera=False, timeout=120):
    """Alternativa con `curl` del sistema (usa el almacén de certificados del sistema; la verificación sigue activa)."""
    import subprocess
    cmd = ["curl", "-sL", "--max-time", str(timeout), "-A", USER_AGENT]
    if solo_cabecera:
        r = subprocess.run(cmd + ["-I", url], capture_output=True, text=True)
        cab = [l for l in r.stdout.splitlines() if l.lower().startswith("content-length")]
        return int(cab[-1].split(":")[1]) if cab else None
    subprocess.run(cmd + ["-o", destino, url], check=True)


def plan_descarga(destino="datos/eph", desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA, consultar_tamanos=True):
    """Arma el plan SIN descargar: qué archivos, tamaño (cabecera HTTP HEAD) y cuáles ya están en `destino`."""
    plan = []
    for it in urls_eph(desde, hasta):
        it = dict(it); it["existe"] = os.path.exists(os.path.join(destino, it["nombre"])); it["bytes"] = None
        if consultar_tamanos:
            try:
                with _abrir(it["url"], "HEAD") as r:
                    it["bytes"] = int(r.headers.get("Content-Length", 0)) or None
                    it["tipo"] = r.headers.get("Content-Type", "")
            except Exception as ex:                       # problema de certificados/red: se prueba curl; si no, se informa
                try:
                    it["bytes"] = _curl(it["url"], solo_cabecera=True)
                except Exception as ex2:
                    it["error"] = f"{ex} | curl: {ex2}"
        plan.append(it)
    return pd.DataFrame(plan)


def descargar_eph(destino="datos/eph", desde=DISPONIBLE_DESDE, hasta=DISPONIBLE_HASTA, confirmado=False, pausa=0.5):
    """Descarga las bases usuarias individuales de la EPH desde INDEC.

    `confirmado=False` (por defecto): MODO PRUEBA. Muestra el plan (archivos y tamaño total) y NO descarga nada.
    `confirmado=True`: descarga los que falten. Verifica que cada archivo sea un ZIP real (INDEC devuelve una página HTML
    con código 200 cuando el archivo no existe) y no vuelve a bajar los que ya están.
    """
    os.makedirs(destino, exist_ok=True)
    plan = plan_descarga(destino, desde, hasta)
    faltan = plan[~plan["existe"]]
    total_mb = (faltan["bytes"].fillna(0).sum()) / 1e6
    print(f"EPH: {len(plan)} archivos en el rango; {len(faltan)} por descargar (~{total_mb:.1f} MB). Fuente: {BASE_URL}/")
    if not confirmado:
        print("MODO PRUEBA: no se descargó nada. Para descargar: descargar_eph(..., confirmado=True)")
        return plan
    ok, fallidos = 0, []
    for _, it in faltan.iterrows():
        ruta = os.path.join(destino, it["nombre"]); tmp = ruta + ".part"
        try:
            with _abrir(it["url"]) as r, open(tmp, "wb") as f:
                cab = r.read(4); f.write(cab)
                if cab[:2] != b"PK":
                    raise ValueError("la respuesta no es un ZIP (¿archivo inexistente?)")
                while True:
                    bloque = r.read(1 << 20)
                    if not bloque:
                        break
                    f.write(bloque)
            os.replace(tmp, ruta); ok += 1
            print(f"  ✔ {it['nombre']} ({os.path.getsize(ruta)/1e6:.1f} MB)")
        except Exception as ex:
            if "CERTIFICATE" in str(ex).upper() or "SSL" in str(ex).upper():      # reintento con curl (verificación activa)
                try:
                    _curl(it["url"], destino=tmp)
                    with open(tmp, "rb") as g:
                        if g.read(2) != b"PK":
                            raise ValueError("la respuesta no es un ZIP (¿archivo inexistente?)")
                    os.replace(tmp, ruta); ok += 1
                    print(f"  ✔ {it['nombre']} ({os.path.getsize(ruta)/1e6:.1f} MB, vía curl)")
                    time.sleep(pausa); continue
                except Exception as ex2:
                    ex = ex2
            fallidos.append((it["nombre"], str(ex)))
            if os.path.exists(tmp): os.remove(tmp)
            print(f"  ✘ {it['nombre']}: {ex}")
        time.sleep(pausa)
    print(f"Listo: {ok} descargados, {len(fallidos)} fallidos.")
    return plan


# ================================================================== 2) LECTURA
def find_eph_files(rutas=("datos/eph", "../datos/eph", "../../datos/eph")):
    """Busca bases individuales (usu_individual_T*.txt/.csv, o EPH_usu_*.zip con una base individual adentro)."""
    out = []
    for r in rutas:
        if not os.path.isdir(r):
            continue
        for p in sorted(glob.glob(os.path.join(r, "**", "*"), recursive=True)):
            b = os.path.basename(p).lower()
            if os.path.isfile(p) and re.search(r"usu_individual.*\.(txt|csv)$", b):
                out.append({"path": p, "member": None})
            elif os.path.isfile(p) and b.endswith(".zip") and "eph_usu" in b:
                with zipfile.ZipFile(p) as z:
                    for m in z.namelist():
                        if re.search(r"usu_individual.*\.(txt|csv)$", m.lower()):
                            out.append({"path": p, "member": m})
    return out


def _leer_texto(handle, nombre=""):
    cab = handle.readline() if hasattr(handle, "readline") else ""
    sep = ";" if cab.count(";") >= cab.count(",") else ","       # INDEC usa ';' (se autodetecta)
    handle.seek(0)
    return pd.read_csv(handle, sep=sep, usecols=lambda c: c.strip().upper() in NEEDED, low_memory=False)


def read_individual(src):
    """Lee UNA base individual trimestral (sólo las columnas necesarias). Columnas en MAYÚSCULAS, valores numéricos."""
    if src["member"] is None:
        with open(src["path"], "r", encoding="latin-1", newline="") as f:
            df = _leer_texto(f, src["path"])
    else:
        with zipfile.ZipFile(src["path"]) as z, z.open(src["member"]) as raw:
            df = _leer_texto(io.StringIO(raw.read().decode("latin-1")), src["member"])
    df.columns = [c.strip().upper() for c in df.columns]
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    faltan = OBLIGATORIAS - set(df.columns)
    if faltan:
        raise ValueError(f"Faltan columnas obligatorias {sorted(faltan)} en {src}. Revisar el diseño de registro de INDEC.")
    for opt in ("CAT_OCUP", "PP07H", "PP07I", "CH04", "TRIMESTRE"):
        if opt not in df.columns:
            df[opt] = np.nan
    return df


def read_all(srcs):
    return pd.concat([read_individual(s) for s in srcs], ignore_index=True)


def resumen_base(df):
    """Control rápido tras cargar datos reales: filas por año, % de nulos en variables clave, valores de PP07H."""
    print("Filas por año:", df.groupby("ANO4").size().to_dict())
    print("Nulos (%):", (df[["CH06", "ESTADO", "CAT_OCUP", "PP07H", "PP07I", "PONDERA"]].isna().mean() * 100).round(1).to_dict())
    print("ESTADO (1 ocupado, 2 desocupado, 3 inactivo):", df["ESTADO"].value_counts(dropna=False).sort_index().to_dict())
    print("PP07H entre asalariados (1 sí, 2 no, 9 NS/NR):", df.loc[df["CAT_OCUP"] == 3, "PP07H"].value_counts(dropna=False).sort_index().to_dict())


# ================================================================== 3) FILTRO: ocupados con descuento jubilatorio
def add_flags(df, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None):
    """Población 18-64 con entrevista individual; construye `reg` = prob. de estar aportando.
    Asalariado (CAT_OCUP==3): reg = (PP07H==1). Independiente (1,2): según `ind_rule`. Desocupado/inactivo: 0."""
    d = df[(df["CH06"] >= EDAD_MIN) & (df["CH06"] <= EDAD_MAX) & (df["ESTADO"].isin([1, 2, 3]))].copy()
    if sexo is not None:
        d = d[d["CH04"] == sexo]
    ocupado = d["ESTADO"] == 1
    asal = ocupado & (d["CAT_OCUP"] == 3)
    indep = ocupado & d["CAT_OCUP"].isin([1, 2])
    reg = np.zeros(len(d))
    reg[asal.values] = (d.loc[asal, "PP07H"] == 1).astype(float).values       # 2 (no) y 9 (NS/NR) => no registrado
    if ind_rule == "pp07i":
        reg[indep.values] = (d.loc[indep, "PP07I"] == 1).astype(float).values
    elif ind_rule == "escenario":
        if ind_share is None or not (0 <= ind_share <= 1):
            raise ValueError("ind_rule='escenario' requiere ind_share en [0,1] (supuesto explícito).")
        reg[indep.values] = float(ind_share)
    elif ind_rule != "solo_asalariados":
        raise ValueError(f"ind_rule desconocida: {ind_rule}")
    d["reg"] = reg
    d["edad"] = d["CH06"].astype(int)
    d["cohorte"] = (d["ANO4"] - d["CH06"]).astype(int)        # error de ±1 año (no se usa la fecha exacta)
    d["w"] = d["PONDERA"].astype(float)
    return d[["cohorte", "edad", "w", "reg"]]


# ================================================================== 4) PSEUDO-PANEL + MODELO
def build_cells(d, min_n=MIN_N_CELDA):
    """Densidad ponderada por (cohorte, edad), agregando todos los trimestres/años en que esa cohorte tuvo esa edad."""
    d = d.assign(wreg=d["w"] * d["reg"])
    g = d.groupby(["cohorte", "edad"]).agg(n=("w", "size"), w=("w", "sum"), wreg=("wreg", "sum")).reset_index()
    g["dens"] = g["wreg"] / g["w"]
    return g[g["n"] >= min_n].reset_index(drop=True)


def _sig(x):
    return 1 / (1 + np.exp(-x))


def fit_age_cohort_logit(cells, iters=100, ridge=1e-3):
    """Logit aditivo: logit d_c(a) = alpha_c + f(a). Supuesto: el perfil por edad f(a) es común a todas las cohortes."""
    coh = np.sort(cells["cohorte"].unique()); eds = np.sort(cells["edad"].unique())
    ci = {c: i for i, c in enumerate(coh)}; ei = {a: i for i, a in enumerate(eds)}
    X = np.zeros((len(cells), len(coh) + len(eds) - 1))
    for r, (c, a) in enumerate(zip(cells["cohorte"], cells["edad"])):
        X[r, ci[c]] = 1.0
        if ei[a] > 0:
            X[r, len(coh) + ei[a] - 1] = 1.0
    y = cells["dens"].values; m = cells["n"].values.astype(float)
    yy = np.clip(y, 0.01, 0.99)
    beta = np.linalg.solve(X.T @ (m[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (m * np.log(yy / (1 - yy))))
    for _ in range(iters):
        eta = X @ beta
        mu = np.clip(_sig(eta), 1e-6, 1 - 1e-6)
        W = m * mu * (1 - mu)
        z = eta + (y - mu) / (mu * (1 - mu))
        new = np.linalg.solve(X.T @ (W[:, None] * X) + ridge * np.eye(X.shape[1]), X.T @ (W * z))
        if np.max(np.abs(new - beta)) < 1e-9:
            beta = new; break
        beta = new
    alpha = pd.Series(beta[:len(coh)], index=coh)
    f = pd.Series(np.r_[0.0, beta[len(coh):]], index=eds)
    n_ed = cells.groupby("cohorte")["edad"].nunique().reindex(coh)
    return {"alpha": alpha, "f": f, "n_edades": n_ed, "cells": cells}


def extrapolar_alpha(fit, cohortes, modo="ultimas_k", k=5, ventana_tendencia=15):
    """Efecto de cohorte para cohortes con pocos datos (muy jóvenes, muy viejas o futuras). Supuesto explícito."""
    ok = fit["alpha"][fit["n_edades"] >= MIN_EDADES_COHORTE]
    out = {}
    for c in cohortes:
        if c in ok.index:
            out[c] = ok[c]
        elif c > ok.index.max():
            if modo == "ultimas_k":
                out[c] = ok.iloc[-k:].mean()
            elif modo == "tendencia":
                w = ok.iloc[-ventana_tendencia:]; b = np.polyfit(w.index.values, w.values, 1)
                out[c] = np.polyval(b, c)
            else:
                raise ValueError(modo)
        else:
            out[c] = ok.iloc[:k].mean()
    return pd.Series(out)


def densidad_modelada(fit, alpha_c):
    """Matriz cohorte x edad (18..64) de d_c(a). Edades sin datos usan el perfil f(a) del modelo (interpolado)."""
    f = fit["f"].reindex(EDADES).interpolate(limit_direction="both")
    return pd.DataFrame(_sig(alpha_c.values[:, None] + f.values[None, :]), index=alpha_c.index, columns=EDADES)


# ================================================================== 5) EDAD EFECTIVA DE INGRESO AL EMPLEO REGISTRADO
def edad_ingreso_modelada(dens, umbral=0.5):
    """Por cohorte, a partir de la densidad modelada d_c(a):
        meseta   = máx_a d_c(a)
        F_c(a)   = envolvente creciente de d_c(a) / meseta  (se interpreta como "fracción que ya ingresó" a la edad a)
        edad_umbral = primera edad con F >= umbral (por defecto 50 % de la meseta)
        edad_media  = 18 + Σ_{a=18}^{63} (1 − F_c(a))   (edad media de ingreso si F fuera la distribución de la edad de ingreso)
    ES UNA APROXIMACIÓN: la EPH no observa el primer empleo registrado; supone ingreso monótono sin salidas del empleo registrado."""
    v = dens.values
    meseta = v.max(axis=1)
    F = np.clip(np.maximum.accumulate(v, axis=1) / meseta[:, None], 0, 1)
    edad_umbral = np.array([EDADES[np.argmax(F[i] >= umbral)] for i in range(len(F))], dtype=float)
    edad_media = EDAD_MIN + (1 - F[:, :-1]).sum(axis=1)
    return pd.DataFrame({"meseta_densidad": meseta, "edad_ingreso_umbral50": edad_umbral, "edad_ingreso_media": edad_media},
                        index=dens.index)


def edad_ingreso_observada(fit, dens_modelada, umbral=0.5):
    """Estimación EMPÍRICA (sin modelo) de la edad de ingreso: primera edad OBSERVADA en que la densidad de la cohorte
    llega al `umbral` de su meseta modelada. Sólo para cohortes observadas desde edades tempranas; NaN en el resto."""
    cells = fit["cells"]; out = {}
    meseta = dens_modelada.max(axis=1)
    for c, g in cells.groupby("cohorte"):
        if c not in meseta.index:
            continue
        g = g.sort_values("edad")
        if g["edad"].min() > EDAD_MIN + 2:            # no vimos los años de entrada
            out[c] = np.nan; continue
        hit = g[g["dens"] >= umbral * meseta[c]]
        out[c] = float(hit["edad"].iloc[0]) if len(hit) else np.nan
    return pd.Series(out, name="edad_ingreso_observada").reindex(dens_modelada.index)


# ================================================================== 6) Ā
def supervivencia_gompertz(s65, b=GOMPERTZ_B):
    """S(a) desde el nacimiento para a=18..64 (Gompertz con pendiente b calibrada para que S(65)=s65 por cohorte)."""
    s65 = np.asarray(s65, float)
    a = -np.log(s65) * b / (np.exp(65 * b) - 1)
    return np.exp(-(a[:, None] / b) * (np.exp(b * EDADES[None, :]) - 1))


def abar_por_cohorte(dens, s65):
    """Ā(c) = Σ_{a=18}^{64} S_c(a) · d_c(a)  (incluye a quienes nunca aportan: densidad 0)."""
    S = supervivencia_gompertz(s65.reindex(dens.index).values)
    return pd.Series((S * dens.values).sum(axis=1), index=dens.index, name="A_bar"), S


def pipeline(df_ind, s65, ind_rule=IND_RULE_DEFAULT, ind_share=IND_SHARE_DEFAULT, sexo=None,
             modo_extrap="ultimas_k", cohortes=None):
    """EPH individual -> tabla por cohorte (Ā y edad de ingreso). Igual para datos sintéticos y reales."""
    d = add_flags(df_ind, ind_rule, ind_share, sexo)
    cells = build_cells(d)
    fit = fit_age_cohort_logit(cells)
    cohortes = list(cohortes) if cohortes is not None else list(s65.index)
    alpha_c = extrapolar_alpha(fit, cohortes, modo=modo_extrap)
    dens = densidad_modelada(fit, alpha_c)
    abar, S = abar_por_cohorte(dens, s65)
    ing = edad_ingreso_modelada(dens)
    tabla = pd.DataFrame({"A_bar": abar,
                          "n_edades_obs": fit["n_edades"].reindex(abar.index).fillna(0).astype(int),
                          "S65": s65.reindex(abar.index),
                          "A_max_sin_densidad": S.sum(axis=1)}).join(ing)
    tabla["edad_ingreso_observada"] = edad_ingreso_observada(fit, dens)
    return {"tabla": tabla, "dens": dens, "fit": fit, "cells": cells}


def estimar_abar(df_ind, s65, cohortes=None, modo_extrap="ultimas_k", ind_share_escenario=0.5):
    """Ā por cohorte con las TRES reglas para independientes. Devuelve un DataFrame (una fila por cohorte):
        A_bar_solo_asalariados (cota inferior) | A_bar_pp07i | A_bar_indep50 (escenario) |
        edad de ingreso (modelada / observada) | n_edades_obs | S65"""
    reglas = {"solo_asalariados": dict(ind_rule="solo_asalariados"),
              "pp07i": dict(ind_rule="pp07i"),
              f"indep{int(round(ind_share_escenario * 100))}": dict(ind_rule="escenario", ind_share=ind_share_escenario)}
    out, base = {}, None
    for nombre, kw in reglas.items():
        r = pipeline(df_ind, s65, cohortes=cohortes, modo_extrap=modo_extrap, **kw)
        chequeos(r)
        out[f"A_bar_{nombre}"] = r["tabla"]["A_bar"]
        if base is None:
            base = r["tabla"][["n_edades_obs", "S65", "meseta_densidad", "edad_ingreso_umbral50", "edad_ingreso_media", "edad_ingreso_observada"]]
    return pd.DataFrame(out).join(base)


def chequeos(res):
    t, dens, cells = res["tabla"], res["dens"], res["cells"]
    assert np.all((cells["dens"] >= 0) & (cells["dens"] <= 1)), "densidad observada fuera de [0,1]"
    assert np.all((dens.values >= 0) & (dens.values <= 1)), "densidad modelada fuera de [0,1]"
    assert np.all((t["A_bar"] >= 0) & (t["A_bar"] <= 47)), "A_bar fuera de [0,47]"
    assert np.all(t["A_bar"] <= t["A_max_sin_densidad"] + 1e-9), "A_bar supera Σ S(a) (densidad = 1)"
    assert np.all((t["edad_ingreso_umbral50"] >= EDAD_MIN) & (t["edad_ingreso_umbral50"] <= EDAD_MAX)), "edad de ingreso fuera de rango"
    assert np.all((t["edad_ingreso_media"] >= EDAD_MIN) & (t["edad_ingreso_media"] <= EDAD_MAX)), "edad media de ingreso fuera de rango"
    a2, _ = abar_por_cohorte(dens, (t["S65"] + 0.02).clip(upper=0.999))
    assert np.all(a2.values > t["A_bar"].values - 1e-12), "monotonía en S65 violada"
    return True


# ================================================================== 7) CONEXIÓN CON EL TBP
def tbp_con_abar(d, abar, tau=0.2177, rho=0.403, col_abar="A_bar_solo_asalariados"):
    """TBP_C y TBP_A usando Ā(t) por cohorte (en vez de un valor único).
    `d`: DataFrame del notebook principal con columnas cohorte, N_t, S65, J, E65. `abar`: salida de estimar_abar (o Serie)."""
    a = abar[col_abar] if isinstance(abar, pd.DataFrame) else abar
    x = d.set_index("cohorte").join(a.rename("A_bar_eph"), how="left")
    x["TBP_C_Abar"] = x["N_t"] * x["S65"] * x["A_bar_eph"] * tau / (x["J"] * rho)
    x["TBP_A_Abar"] = x["TBP_C_Abar"] / x["E65"]
    return x.reset_index()


# ================================================================== 8) DATOS SINTÉTICOS Y AUTOTEST
def banner(txt="DATOS SINTÉTICOS — no son Argentina"):
    line = "#" * 78
    print("\n".join([line, "##" + txt.center(74) + "##", line]))


def _f_edad_sint(a):
    return 1.2 * np.exp(-((np.asarray(a) - 40) / 18.0) ** 2) - 1.0


def _alpha_sint(c):
    return -0.2 + 0.008 * (np.asarray(c) - 1970)


def _s65_sint(c):
    return np.clip(0.70 + 0.0035 * (np.asarray(c) - 1950), 0.5, 0.97)


def generar_eph_sintetica(anios=range(2003, 2025), n_por_edad=25, seed=123):
    """EPH de juguete con las mismas columnas que la real. NO son datos de Argentina."""
    rng = np.random.default_rng(seed); filas = []
    for y in anios:
        for q in (1, 2, 3, 4):
            for a in range(14, 71):
                p = _sig(_alpha_sint(y - a) + _f_edad_sint(a) + 0.10 * np.sin((y - 2003) / 3.0))
                reg = rng.random(n_por_edad) < p
                v = rng.random(n_por_edad)
                estado = np.where(reg, 1, np.where(v < 0.45, 1, 3))
                cat = np.where(estado == 1, np.where(reg | (v < 0.30), 3, 2), np.nan)
                filas.append(pd.DataFrame({"ANO4": y, "TRIMESTRE": q, "CH04": rng.integers(1, 3, n_por_edad), "CH06": a,
                                           "ESTADO": estado, "CAT_OCUP": cat,
                                           "PP07H": np.where(cat == 3, np.where(reg, 1, 2), np.nan),
                                           "PP07I": np.where(cat == 2, 2, np.nan),
                                           "PONDERA": rng.integers(300, 900, n_por_edad)}))
    return pd.concat(filas, ignore_index=True)


def autotest(verbose=True):
    """Prueba de punta a punta con datos SINTÉTICOS: pipeline, lector (txt y zip), filtros y conexión con el TBP."""
    if verbose: banner()
    df = generar_eph_sintetica()
    coh = np.arange(1940, 2036); s65 = pd.Series(_s65_sint(coh), index=coh)
    tabla = estimar_abar(df, s65, cohortes=coh)
    # verdad sintética
    dens_true = pd.DataFrame(_sig(_alpha_sint(coh)[:, None] + _f_edad_sint(EDADES)[None, :]), index=coh, columns=EDADES)
    a_true, _ = abar_por_cohorte(dens_true, s65)
    obs = tabla[tabla["n_edades_obs"] >= MIN_EDADES_COHORTE]
    # la regla "solo_asalariados" es la que coincide con cómo se generó el dato sintético (todos los registrados son asalariados)
    mae = (obs["A_bar_solo_asalariados"] - a_true.reindex(obs.index)).abs().mean()
    assert mae < 1.5, f"el pipeline no recupera la verdad sintética (MAE={mae:.2f})"
    assert (tabla["A_bar_solo_asalariados"] <= tabla["A_bar_pp07i"] + 1e-9).all(), "pp07i no puede dar menos que solo_asalariados"
    assert (tabla["A_bar_pp07i"] <= tabla["A_bar_indep50"] + 1e-9).all(), "indep50 no puede dar menos que pp07i"
    assert tabla["edad_ingreso_media"].between(18, 64).all() and tabla["edad_ingreso_umbral50"].between(18, 64).all()
    # lector: txt y zip con el nombre típico de INDEC
    with tempfile.TemporaryDirectory() as tmp:
        sub = df[df["ANO4"].isin([2010, 2011])].copy()
        sub.columns = [c.lower() if c in ("CH04", "CH06", "ESTADO") else c for c in sub.columns]
        p = os.path.join(tmp, "usu_individual_T110.txt"); sub.to_csv(p, sep=";", index=False)
        s1 = find_eph_files([tmp]); assert len(s1) == 1 and len(read_all(s1)) == len(sub)
        zp = os.path.join(tmp, "EPH_usu_1_Trim_2010_txt.zip")
        with zipfile.ZipFile(zp, "w") as z:
            z.write(p, "usu_individual_T110.txt")
        os.remove(p)
        s2 = find_eph_files([tmp]); assert len(s2) == 1 and s2[0]["member"] is not None and len(read_all(s2)) == len(sub)
    assert find_eph_files(["/ruta/que/no/existe"]) == []
    # conexión con el TBP: con Ā(t) constante debe reproducir la fórmula con un valor único
    d = pd.DataFrame({"cohorte": [2000, 2001], "N_t": [700000., 690000.], "S65": [0.88, 0.89], "J": [1.2e7, 1.2e7], "E65": [21., 21.]})
    fijo = pd.Series(30.0, index=[2000, 2001])
    r = tbp_con_abar(d, fijo)
    assert np.allclose(r["TBP_C_Abar"], d["N_t"] * d["S65"] * 30 * 0.2177 / (d["J"] * 0.403))
    if verbose:
        print(f"[sintético] error absoluto medio de Ā: {mae:.3f} años | cohortes observadas: {len(obs)}")
        print("Autotest OK (pipeline, 3 reglas, edad de ingreso, lector txt/zip, conexión con TBP).")
        banner("FIN AUTOTEST — estos números NO son Argentina")
    return tabla

### 12.2 Descarga de las bases (modo prueba por defecto)

La fuente es INDEC (`https://www.indec.gob.ar/ftp/cuadros/menusuperior/eph/`). Mientras `ACEPTO_DESCARGA` sea `False` **no se baja nada**: sólo se consulta el tamaño de cada archivo. Cambiarlo a `True` descarga los que falten a `datos/eph/` (esa carpeta no se sube al repositorio).

In [ ]:
DESTINO_EPH = "datos/eph"
RUTAS_EPH = ["datos/eph", "../datos/eph", "../../datos/eph"]
ACEPTO_DESCARGA = False          # <-- poner True para descargar (35 archivos, ~107 MB)
plan_eph = descargar_eph(DESTINO_EPH, desde=(2017, 2), hasta=(2025, 4), confirmado=ACEPTO_DESCARGA)
display(plan_eph[["anio", "trim", "nombre", "bytes", "existe"]].head(4))
print("... total del plan:", round(plan_eph["bytes"].fillna(0).sum() / 1e6, 1), "MB en", len(plan_eph), "archivos")

### 12.3 Carga y chequeo de las bases

Si no hay archivos en `datos/eph/`, el notebook sigue de largo y avisa que **faltan datos** (no inventa nada).

In [ ]:
srcs_eph = find_eph_files(RUTAS_EPH)
HAY_EPH = len(srcs_eph) > 0
if not HAY_EPH:
    print("=" * 78)
    print("FALTAN DATOS: no hay bases individuales de la EPH en", RUTAS_EPH)
    print("Para correr con datos reales: ACEPTO_DESCARGA = True (celda 12.2) o dejar los .zip de INDEC en datos/eph/.")
    print("No se calculó ningún Ā para Argentina.")
    print("=" * 78)
else:
    df_eph = read_all(srcs_eph)
    print(f"Archivos: {len(srcs_eph)} | filas: {len(df_eph):,} | años: {int(df_eph['ANO4'].min())}-{int(df_eph['ANO4'].max())}")
    resumen_base(df_eph)

### 12.4 $\bar A$ por cohorte y edad efectiva de ingreso

$$\bar A_c=\sum_{a=18}^{64} S_c(a)\; d_c(a)$$

con $S_c(a)$ la supervivencia de la cohorte hasta la edad $a$ (calibrada con $S_{65}$ de cada cohorte) y $d_c(a)$ la densidad de aportes (proporción registrada, incluye ceros). La **edad efectiva de ingreso** se estima de dos maneras: *modelada* (18 + suma de $1-F_c(a)$, con $F_c$ la fracción de la meseta de densidad ya alcanzada) y *observada* (primera edad observada en que la densidad llega al 50 % de su meseta; sólo para cohortes vistas desde edades tempranas). Son aproximaciones: la EPH no observa el primer empleo registrado.

In [ ]:
if HAY_EPH:
    s65_real = d.set_index("cohorte")["S65"]
    abar_eph = estimar_abar(df_eph, s65_real, cohortes=s65_real.index)
    os.makedirs(OUT, exist_ok=True)
    abar_eph.to_csv(f"{OUT}/abar_eph_por_cohorte.csv")
    display(abar_eph.loc[[c for c in (1960, 1970, 1980, 1990, 2000, 2005) if c in abar_eph.index]].round(2))
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    for col in [c for c in abar_eph.columns if c.startswith("A_bar_")]:
        ax[0].plot(abar_eph.index, abar_eph[col], label=col.replace("A_bar_", "regla: "))
    ax[0].axhline(14.2, color="gray", ls=":", label="14,2 (ANSES, sesgado)"); ax[0].axhline(30, color="gray", ls="--", label="30 (requisito legal)")
    ax[0].set_title("Ā por cohorte (EPH)"); ax[0].set_xlabel("cohorte"); ax[0].set_ylabel("años"); ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_media"], label="modelada (media)")
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_umbral50"], label="modelada (umbral 50 %)")
    ax[1].plot(abar_eph.index, abar_eph["edad_ingreso_observada"], "o", ms=3, label="observada (umbral 50 %)")
    ax[1].set_title("Edad efectiva de ingreso al empleo registrado"); ax[1].set_xlabel("cohorte"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Sin datos EPH: nada que calcular (ver 12.3). La prueba sintética está en 12.6.")

### 12.5 Conexión con el TBP: $\bar A(t)$ por cohorte

Con $\bar A$ distinto por cohorte, $TBP_C(t)=\dfrac{N_t(1-\mu)\,\bar A(t)\,\tau}{J\,\rho}$ **ya no es proporcional** a un único valor: **cambia la forma de la curva**, no sólo el nivel (a diferencia de las bandas de la sección 7).

In [ ]:
if HAY_EPH:
    tb = tbp_con_abar(d, abar_eph, tau=TAU, rho=RHO, col_abar="A_bar_solo_asalariados")
    fig, ax = plt.subplots(1, 2, figsize=(13, 4))
    ax[0].plot(d.cohorte, d.TBP_C_legal, label="Ā = 30 (constante)"); ax[0].plot(d.cohorte, d.TBP_C_obs, label="Ā = 14,2 (constante)")
    ax[0].plot(tb.cohorte, tb.TBP_C_Abar, label="Ā(t) EPH (solo asalariados)")
    ax[0].set_title("TBP_C por cohorte"); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
    base = tb.loc[tb.cohorte == 1960, "TBP_C_Abar"].iloc[0]
    ax[1].plot(d.cohorte, 100 * d.TBP_C_legal / d.TBP_C_legal[d.cohorte == 1960].iloc[0], label="Ā constante (la forma no cambia)")
    ax[1].plot(tb.cohorte, 100 * tb.TBP_C_Abar / base, label="Ā(t) EPH")
    ax[1].set_title("Índice (cohorte 1960 = 100)"); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()
else:
    print("Sin datos EPH: no se calcula TBP con Ā(t).")

### 12.6 Prueba con datos sintéticos (autotest)

Verifica la rutina de punta a punta con una EPH **inventada**: pipeline, tres reglas, edad de ingreso, lector de `.txt` y `.zip` y conexión con el TBP. **No son datos de Argentina.**

In [ ]:
_tabla_sintetica = autotest()

### 12.7 Lectura y límites

* **Qué se puede afirmar hoy:** que la rutina está probada con datos sintéticos y que las bases de la EPH 2017-T2 a 2025-T4 existen y se pueden bajar. **No** se puede afirmar ningún valor de $\bar A$ para Argentina hasta correrla con los microdatos.
* **Cobertura:** la EPH es urbana (aglomerados) y desde 2017 sólo hay 9 años de datos: cada cohorte se observa en pocas edades y el resto se completa con el perfil por edad común a todas (supuesto fuerte; período, edad y cohorte no se separan del todo).
* **`PP07H`:** pregunta a asalariados y mide la semana de referencia, no el año. Un descuento no equivale automáticamente a un año de aporte computable (moratorias, cajas provinciales).
* **Cuentapropistas:** no hay una variable inequívoca de aporte en la base individual; por eso se reportan tres reglas (cota inferior, `PP07I`, escenario). La verdadera estará entre ellas.
* **Edad de ingreso:** es una aproximación (ingreso monótono, sin salidas del empleo registrado); no es la edad del primer empleo.
* **Comparabilidad:** el $\bar A$ de la EPH no es comparable uno a uno con el 14,2 de ANSES (piso de 12 meses y registros desde jul-1994).
* **Verificar con la primera descarga:** nombres de variables y separador del `.txt` (`resumen_base` ayuda a detectar problemas).